# NB10b: Faithful Paper Replications + Optuna on Faithful Protocol -- V3 per-pixel v10

> Copyright (C) 2024-2026 Marco Heinzen - SPDX-License-Identifier: AGPL-3.0-or-later
> Part of the Master Thesis "Building Damage Assessment with Multimodal Satellite Time Series and Machine Learning in the Russia-Ukraine War 2022-2026"
> Code hosted at https://github.com/marcoheinzen/bda
> Parts of this code were written or improved with the assistance of Claude (Anthropic); all other code, and the concept, research, architecture, design, execution, testing and validation throughout, are the author's work.


This notebook replicates four published BDA methods exactly as the
papers describe them, on V3 (per-pixel) data with GroupKFold by city
across 21 UNOSAT-labelled Ukrainian cities. Mirrors NB10a's structure:
**first faithful with paper params, then Optuna on the same protocol.**

## Layout

| Section | Cells | Method | Trial budget |
|---|---|---|---|
| 1 Faithful | B1 + B1_bldg | Ballinger 2024 PWTT (Welch t-test, parameter-free) | -- |
| 2 Faithful + Optuna | B2 + B2_bldg + B2_Optuna | Scher 2025 CCD; Optuna tunes (k, z_thresh, persistence) | 200 / pol |
| 3 Faithful | B3a + B3a_bldg | Aimaiti 2022 log-ratio (parameter-free at AUC) | -- |
| 4 Faithful + ablation + Optuna | B5 + B5_bldg + B6 + B5_Optuna | Ahmad UISEM Model 2 faithful; Model 1 ablation; per-classifier Optuna + tuned voting | 200 / classifier |

B1 and B3a have no learnable hyperparameters in the paper, so no
Optuna companion. B2 has three (k, z_thresh, persistence_months) -- the
parameters Scher's own sensitivity analysis varies. B5 has the standard
ML hyperparameters of the four sub-classifiers.

## Out of scope (-> NB11b v2)

- Different sampling unit (V2 footprint instead of V3 pixel) -> NB11b
  C_V2_BEST, C_V2_F8_UISEM
- Modality combinations the paper did not target
- Cross-paper engineering ceiling

# CONFIG + GLOBAL SETUP

`TIER_SELECTION = [0,1,2]` selects the 21 UNOSAT-labelled cities.
`DATASET_VERSION = 'v3'` is fixed because all four papers operate at
pixel level. `OPTUNA_TRIALS_DEFAULT = 200` for the Optuna-on-faithful
companion cells (B2_Optuna, B5_Optuna).


In [1]:
# @title CELL 1: NB10b CONFIG
TIER_SELECTION = [0,1,2]
CITY_SELECTION = None
REQUIRE_UNOSAT = True
RANDOM_STATE = 42

# Dataset version. NB10b is V3 (per-point single-pixel) only because the four
# replicated papers (Ballinger, Scher, Aimaiti, Ahmad) all operate at pixel
# level. V2 (per-footprint zonal) ceilings are in NB11b.
DATASET_VERSION = 'v3'

# Optuna budget for B2_Optuna and B5_Optuna companion cells (200 default).
OPTUNA_TRIALS_DEFAULT = 200

import platform, os
if platform.system() == "Windows":
    _setup = r"F:\PROJECTS\masterthesis\gdrive\masterthesis\notebooks\global_setup.py"
elif os.path.exists("/content/drive_f"):
    _setup = "/content/drive_f/masterthesis/notebooks/global_setup.py"
else:
    _setup = "/mnt/f/PROJECTS/masterthesis/gdrive/masterthesis/notebooks/global_setup.py"
with open(_setup) as f:
    exec(f.read())


BDA GLOBAL SETUP
Started: 2026-05-07 23:42:47
Python: 3.12.12

[1/7] Directory Structure
----------------------------------------------------------------------


/home/alpineobotics/miniconda3/envs/bda/lib/python3.12/site-packages/pyproj/network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)


  GDrive (G:):       /content/drive_f/masterthesis OK
  GDrive (F:):       /content/drive_f/masterthesis OK
  Local data (G:):   /content/masterthesis_local/data OK
  Data stack (F:):   /mnt/f/PROJECTS/masterthesis/data_stack OK

  TIER_SELECTION: [0, 1, 2]
  CITY_SELECTION: None (tier filter)
  REQUIRE_UNOSAT: True
  CITIES_TO_PROCESS: 21 cities

[2/7] Credentials
----------------------------------------------------------------------
  Copernicus: inf***
  OpenTopography: OK
  Earthdata: marcoheinzen

[3/7] Python Packages
----------------------------------------------------------------------


<string>:564: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.



  Already installed: 23
  Newly installed:   0
  Failed:            0

[4/7] Global Imports & Configuration
----------------------------------------------------------------------
  All imports loaded

[5/7] Processing Config & SNAP
----------------------------------------------------------------------
  GPT: Usage:
  Temporal baseline: 10-24 days
  Wavelength: 0.0555

[6/7] GPU Status
----------------------------------------------------------------------
  CUDA available: NVIDIA GeForce RTX 2070 SUPER
    CUDA version: 12.8

[7/7] Disk Space
----------------------------------------------------------------------
  GDrive (G:)     910.9/7452.0 GB (6541.1 GB free)
  GDrive (F:)     1408.1/3726.0 GB (2317.9 GB free)
  Local data      11559.5/14901.9 GB (3342.4 GB free)
  Data stack      1408.1/3726.0 GB (2317.9 GB free)
  WSL ext4        68.6/1006.9 GB (887.1 GB free)

GLOBAL SETUP COMPLETE
  Torch device: cuda
  Cities: 21, CITY=Avdiivka
  Functions: load_aoi(), load_aoi_gdf(), load_aoi_

# CELL S0: SHARED HELPERS

Loads V3 parquet manifest, point metadata, and defines:
- `load_v3(manifest_key)` -- read a V3 parquet by manifest key, merge damage labels.
- `prepare_Xy_from(df, feat_cols)` -- median-imputed X, y, groups, point_ids, clean cols.
- `evaluate_groupkfold(clf, X, y, groups, ...)` -- OOF AUC + F1 with N-fold GroupKFold.
- `build_uisem(include_adaboost)` -- VotingClassifier (RF + GB + XGB [+ AB]).
- `save_oof_from_result(...)` -- canonical OOF parquet for NB12 stacking.
- `save_oof_unsupervised(...)` -- rank-normalized scores for unsupervised methods.
- `log_result(...)` -- ResultRegistry log.

All helpers mirror NB09b v5 / NB09bV3 v2 conventions but use V3 column names
(`point_id`, bare `s1__vv` / `s1__vh` / `s1__coh_vv`, etc.) directly without
falling back to non-existent V2-style `__mean` suffixed columns.

In [2]:
# @title CELL S0: NB10b SHARED HELPERS
import sys, importlib, re, time, gc, json
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                              ExtraTreesClassifier, VotingClassifier, AdaBoostClassifier)
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score
from sklearn.impute import SimpleImputer
from sklearn.base import clone
from scipy import stats as scipy_stats
from scipy.stats import rankdata as _rankdata
import warnings
warnings.filterwarnings("ignore")

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False

from lightgbm import LGBMClassifier

print("=" * 70)
print("CELL S0: NB10b SHARED HELPERS")
print("=" * 70)

if str(NOTEBOOKS_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS_DIR))

# --- V3 dataset path (V3 not in global_setup.py; defined locally) ----
DATASET_ROOT_V3 = STACK_DIR / 'dataset' / 'v3'
MANIFEST_V3_PATH = DATASET_ROOT_V3 / 'parquet_manifest.json'
if not MANIFEST_V3_PATH.exists():
    raise FileNotFoundError(f"V3 manifest not found: {MANIFEST_V3_PATH}")
with open(MANIFEST_V3_PATH) as _f:
    MANIFEST = json.load(_f)
print(f"  V3 manifest: {MANIFEST_V3_PATH}")
print(f"  Parquets registered: {len(MANIFEST['parquets'])}")

# --- load V3 point metadata ------------------------------------------
_tiers = TIER_SELECTION if isinstance(TIER_SELECTION, list) else [0, 1, 2]
_pts_pattern = str(DATASET_ROOT_V3 / 'bda_points_t{tier}.parquet')
df_points = load_tier_parquets(_pts_pattern, _tiers)

CITIES_TO_PROCESS, _battle_dates = resolve_cities(
    tier_selection=TIER_SELECTION,
    city_selection=[CITY_SELECTION] if isinstance(CITY_SELECTION, str) else CITY_SELECTION,
    require_unosat=REQUIRE_UNOSAT,
)
df_points = df_points[df_points['city'].isin(CITIES_TO_PROCESS)].copy()
df_points = df_points[df_points['damage_binary'] >= 0].copy()
print(f"  Cities: {len(CITIES_TO_PROCESS)}")
print(f"  Points: {len(df_points)} (damaged={int((df_points['damage_binary']==1).sum())}, undamaged={int((df_points['damage_binary']==0).sum())})")

SAMPLE_KEY = 'point_id'
TARGET_COL = 'damage_binary'
N_FOLDS = 5
EXPERIMENT_LOG = {}

# --- paper reference numbers ----------------------------------------
DIETRICH_PAPER = {'auc': 0.813, 'f1': 0.749, 'precision': 0.671, 'recall': 0.846}
BALLINGER_PAPER = {'auc': 0.757, 'f1': 0.684, 'recall': 0.763}
SCHER_PAPER = {'tpr': 0.591, 'fpr': 0.0114, 'f1': 0.68, 'k': -0.05, 'z_thresh': -2.0, 'persistence_months': 3}
AIMAITI_PAPER = {'tpr_all': 0.58, 'tpr_large': 0.76, 'min_footprint_m2': 300}
AHMAD_PAPER = {'overall_acc': 0.92, 'f1': 0.92, 'precision': 0.88, 'recall': 0.96, 'n_features_best': 27}

# --- experiment id for OOF parquets ---------------------------------
import hashlib as _hashlib, datetime as _dt
EXPERIMENT_ID = _dt.datetime.now().strftime('%Y%m%d_%H%M%S') + '_' + _hashlib.md5(
    f'nb10b_{TIER_SELECTION}_{REQUIRE_UNOSAT}_{DATASET_VERSION}'.encode()).hexdigest()[:6]
print(f"  EXPERIMENT_ID: {EXPERIMENT_ID}")

OOF_DIR = (RESULTS_ROOT / 'nb10b_v3' / 'oof_predictions')
OOF_DIR.mkdir(parents=True, exist_ok=True)

# --- leakage filter (same as NB09b) ---------------------------------
import metadata_filter
importlib.reload(metadata_filter)
from metadata_filter import is_non_feature

def drop_leakage(cols):
    return [c for c in cols if not is_non_feature(c)]


# --- V3 manifest-driven loader --------------------------------------
def load_v3(manifest_key, tiers=None, columns=None, sample_frac=None):
    '''Load a V3 parquet by manifest key, merge with damage labels, return df.

    V3 parquets index by point_id rather than building_id. Damage labels live
    in bda_points and are joined here when not already present.
    '''
    if manifest_key not in MANIFEST['parquets']:
        raise KeyError(f"Manifest key not found: {manifest_key}")
    info = MANIFEST['parquets'][manifest_key]
    fname_tmpl = Path(info['pattern']).name
    tiers = tiers if tiers is not None else _tiers

    frames = []
    for t in tiers:
        pf = DATASET_ROOT_V3 / fname_tmpl.format(tier=t)
        if not pf.exists():
            print(f"  WARNING: {pf.name} not found, skipping tier {t}")
            continue
        if columns is not None:
            join_cols = info.get('join_keys', ['city', 'point_id'])
            read_cols = list(dict.fromkeys(join_cols + list(columns)))
            try:
                df_t = pd.read_parquet(pf, columns=read_cols)
            except Exception:
                df_t = pd.read_parquet(pf)
                keep = [c for c in read_cols if c in df_t.columns]
                df_t = df_t[keep]
        else:
            df_t = pd.read_parquet(pf)
        if sample_frac is not None and 0 < sample_frac < 1:
            df_t = df_t.sample(frac=sample_frac, random_state=RANDOM_STATE)
        frames.append(df_t)
    if not frames:
        raise FileNotFoundError(f"No V3 parquets found for key={manifest_key} tiers={tiers}")
    df = pd.concat(frames, ignore_index=True)
    del frames

    df = df.loc[:, ~df.columns.duplicated()]
    df = df[df['city'].isin(CITIES_TO_PROCESS)].copy()
    if TARGET_COL not in df.columns:
        df = df.merge(df_points[['point_id', 'city', TARGET_COL]].drop_duplicates(),
                      on=['point_id', 'city'], how='inner')
    df = df[df[TARGET_COL] >= 0].copy()
    df = df.loc[:, ~df.columns.duplicated()]
    print(f"  load_v3('{manifest_key}'): {len(df):,} rows, {df.shape[1]} cols, {df['city'].nunique()} cities")
    return df


# --- shared eval helpers --------------------------------------------
def prepare_Xy_from(src_df, feat_cols):
    '''Extract X, y, groups, point_ids from a given V3 df.

    Drops all-NaN cols, median-imputes the rest. Returns (X, y, groups,
    point_ids, clean_feat_cols) or (None, None, None, None, None) if
    fewer than 2 features survive leakage / NaN filtering.
    '''
    avail = [c for c in feat_cols if c in src_df.columns]
    avail = drop_leakage(avail)
    if len(avail) < 2:
        return None, None, None, None, None
    df_valid = src_df[src_df[TARGET_COL] >= 0].copy()
    row_mask = ~df_valid[avail].isna().all(axis=1)
    df_sub = df_valid[row_mask]
    nan_col = df_sub[avail].isna().all()
    clean = [c for c in avail if not nan_col[c]]
    if len(clean) < 2:
        return None, None, None, None, None
    imp = SimpleImputer(strategy='median')
    X = imp.fit_transform(df_sub[clean].values)
    y = df_sub[TARGET_COL].values
    groups = df_sub['city'].values
    point_ids = df_sub[SAMPLE_KEY].values
    return X, y, groups, point_ids, clean


def evaluate_groupkfold(clf, X, y, groups, label="", n_folds=N_FOLDS, point_ids=None):
    n_cities = len(np.unique(groups))
    n_folds_actual = min(n_folds, n_cities)
    if n_folds_actual < 2:
        print(f"    SKIP {label}: only {n_cities} cities")
        return None
    gkf = GroupKFold(n_splits=n_folds_actual)
    y_proba_oof = np.full(len(y), np.nan)
    fold_id = np.full(len(y), -1, dtype=int)
    fold_aucs = []
    for fold_idx, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups)):
        clf_copy = clone(clf)
        clf_copy.fit(X[train_idx], y[train_idx])
        if hasattr(clf_copy, 'predict_proba'):
            y_proba_oof[test_idx] = clf_copy.predict_proba(X[test_idx])[:, 1]
        else:
            raw = clf_copy.decision_function(X[test_idx])
            y_proba_oof[test_idx] = 1.0 / (1.0 + np.exp(-raw))
        fold_id[test_idx] = fold_idx
        if len(np.unique(y[test_idx])) > 1:
            fold_aucs.append(roc_auc_score(y[test_idx], y_proba_oof[test_idx]))
    valid = ~np.isnan(y_proba_oof)
    auc = roc_auc_score(y[valid], y_proba_oof[valid])
    f1 = f1_score(y[valid], (y_proba_oof[valid] >= 0.5).astype(int))
    pid_v = (np.asarray(point_ids)[valid] if point_ids is not None
             else np.arange(len(y))[valid].astype(str))
    res = {'auc': auc, 'f1': f1, 'auc_std': np.std(fold_aucs), 'fold_aucs': fold_aucs,
           'y_true': y[valid], 'y_proba': y_proba_oof[valid], 'groups': groups[valid],
           'point_id': pid_v, 'fold_id': fold_id[valid],
           'n_features': X.shape[1], 'experiment': label}
    EXPERIMENT_LOG[label] = res
    print(f"    {label:45s}: AUC={auc:.3f} (+/-{np.std(fold_aucs):.3f})  F1={f1:.3f}")
    return res


def build_uisem(include_adaboost=False):
    '''Ahmad 2024 UISEM ensemble. Defaults to include_adaboost=False because
    Ahmad's actual best Model 2 (92% acc) excludes AdaBoost; Model 1 (with AB)
    scored 91%. Ahmad uses sklearn GradientBoostingClassifier; LGBM is
    substituted here for the same gradient-boosting family at ~50-100x speed
    on 600k rows.'''
    estimators = [
        ('rf', RandomForestClassifier(n_estimators=200, min_samples_leaf=3,
                                       random_state=RANDOM_STATE,
                                       n_jobs=-1, class_weight='balanced')),
        ('gb', LGBMClassifier(n_estimators=200, learning_rate=0.1, max_depth=5,
                               num_leaves=31, random_state=RANDOM_STATE,
                               n_jobs=-1, verbose=-1)),
    ]
    if HAS_XGB:
        estimators.append(('xgb', XGBClassifier(n_estimators=200, random_state=RANDOM_STATE,
                                                  scale_pos_weight=5, eval_metric='logloss', verbosity=0)))
    if include_adaboost:
        estimators.append(('ab', AdaBoostClassifier(n_estimators=100, random_state=RANDOM_STATE)))
    return VotingClassifier(estimators=estimators, voting='soft')


# --- OOF persistence ------------------------------------------------
def save_oof_from_result(res, model_id, variant_id='', is_final=False, threshold=0.5):
    '''Supervised: canonical OOF from a result dict.

    Output schema column for V3 is 'point_id' (not 'building_id'). NB12 stacking
    must filter on the dataset version when joining V2 building_id-keyed and
    V3 point_id-keyed OOF parquets.
    '''
    if res is None:
        return None
    needed = ('y_true', 'y_proba', 'groups', 'point_id', 'fold_id')
    if not all(k in res for k in needed):
        print(f"    [save_oof] skipped {model_id}: missing keys")
        return None
    y_proba = np.asarray(res['y_proba'])
    yt = np.asarray(res['y_true']).astype(int)
    fold = np.asarray(res['fold_id']).astype(int)
    pid = np.asarray(res['point_id'])
    cty = np.asarray(res['groups'])
    yp = (y_proba >= threshold).astype(int)
    cm_class = np.where(yt == 1,
                        np.where(yp == 1, 'TP', 'FN'),
                        np.where(yp == 1, 'FP', 'TN'))
    oof_df = pd.DataFrame({
        'point_id':      pid,
        'city':          cty,
        'fold_id':       fold,
        'y_true':        yt,
        'y_proba':       y_proba.astype(float),
        'y_pred':        yp,
        'cm_class':      cm_class,
        'model_id':      model_id,
        'experiment_id': EXPERIMENT_ID,
        'variant_id':    variant_id,
        'is_final':      bool(is_final),
        'dataset_version': DATASET_VERSION,
    })
    out = OOF_DIR / f'oof_{model_id}__{EXPERIMENT_ID}.parquet'
    oof_df.to_parquet(out, index=False)
    n_tp = int((cm_class == 'TP').sum()); n_fn = int((cm_class == 'FN').sum())
    n_fp = int((cm_class == 'FP').sum()); n_tn = int((cm_class == 'TN').sum())
    print(f"    saved oof -> {out.name}  TP={n_tp} FN={n_fn} FP={n_fp} TN={n_tn}")
    return out


def save_oof_unsupervised(point_ids, cities, y_true, scores, model_id,
                          variant_id='', threshold=0.5):
    '''Unsupervised: rank-normalize raw scores to [0,1] then threshold at median.

    fold_id=0 (single-pass evaluation). Above-median rank score -> predicted damaged.
    '''
    scores = np.asarray(scores, dtype=float)
    valid = ~np.isnan(scores)
    if valid.sum() == 0:
        print(f"    [save_oof_unsup] skipped {model_id}: no valid scores")
        return None
    s = scores[valid]
    yt = np.asarray(y_true).astype(int)[valid]
    pid = np.asarray(point_ids)[valid]
    cty = np.asarray(cities)[valid]
    y_proba = _rankdata(s) / len(s)
    yp = (y_proba >= threshold).astype(int)
    cm_class = np.where(yt == 1,
                        np.where(yp == 1, 'TP', 'FN'),
                        np.where(yp == 1, 'FP', 'TN'))
    oof_df = pd.DataFrame({
        'point_id':      pid,
        'city':          cty,
        'fold_id':       np.zeros(len(yt), dtype=int),
        'y_true':        yt,
        'y_proba':       y_proba.astype(float),
        'y_pred':        yp,
        'cm_class':      cm_class,
        'model_id':      model_id,
        'experiment_id': EXPERIMENT_ID,
        'variant_id':    variant_id,
        'is_final':      False,
        'dataset_version': DATASET_VERSION,
    })
    out = OOF_DIR / f'oof_{model_id}__{EXPERIMENT_ID}.parquet'
    oof_df.to_parquet(out, index=False)
    n_tp = int((cm_class == 'TP').sum()); n_fn = int((cm_class == 'FN').sum())
    n_fp = int((cm_class == 'FP').sum()); n_tn = int((cm_class == 'TN').sum())
    print(f"    saved oof -> {out.name}  TP={n_tp} FN={n_fn} FP={n_fp} TN={n_tn}")
    return out


# --- registry -------------------------------------------------------
from bda_results import ResultRegistry
registry = ResultRegistry(RESULTS_ROOT, notebook='NB10b_v3')

def log_result(res, cell_id='', parquet_name='', feature_set_name='',
               classifier_name='', feature_cols=None, groups=None,
               cv_method='GroupKFold', note='', tags=None):
    if res is None:
        return
    registry.log_experiment(
        cell_id=cell_id,
        experiment_name=res.get('experiment', ''),
        parquet_name=parquet_name,
        tier_selection=_tiers,
        classifier_name=classifier_name,
        feature_set_name=feature_set_name,
        feature_cols=feature_cols,
        cv_method=cv_method, n_folds=N_FOLDS, imputation='median',
        y_true=res.get('y_true'), y_proba=res.get('y_proba'),
        groups=res.get('groups', groups),
        note=note, tags=tags or [],
    )


# --- output dir + score collector for B4 / B8 -----------------------
import matplotlib.pyplot as plt
from datetime import datetime as _dt

OUT_DIR = RESULTS_ROOT / 'nb10b_v3'
OUT_DIR.mkdir(parents=True, exist_ok=True)

def save_result(data, name, cell_id, fmt='csv'):
    cell_dir = OUT_DIR / cell_id
    cell_dir.mkdir(parents=True, exist_ok=True)
    ts = _dt.now().strftime('%Y%m%d_%H%M%S')
    if fmt == 'csv' and isinstance(data, pd.DataFrame):
        path = cell_dir / f"{name}_{ts}.csv"
        data.to_csv(path, index=False)
    elif fmt == 'json':
        path = cell_dir / f"{name}_{ts}.json"
        with open(path, 'w') as fh:
            json.dump(data, fh, indent=2, default=str)
    else:
        raise ValueError(f"Unknown fmt={fmt}")
    print(f"  Saved: {path.relative_to(OUT_DIR)} ({path.stat().st_size / 1024:.1f} KB)")
    return path

def save_fig(fig, name, cell_id, dpi=150):
    cell_dir = OUT_DIR / cell_id
    cell_dir.mkdir(parents=True, exist_ok=True)
    ts = _dt.now().strftime('%Y%m%d_%H%M%S')
    path = cell_dir / f"{name}_{ts}.png"
    fig.savefig(path, dpi=dpi, bbox_inches='tight', facecolor='white')
    plt.show()
    print(f"  Plot: {path.relative_to(OUT_DIR)}")
    return path

ALL_SCORES = {}
B0_RESULTS = {}

print(f"  Helpers: load_v3(), prepare_Xy_from(), evaluate_groupkfold(), build_uisem()")
print(f"  Output:  {OUT_DIR}")
print(f"  OOF dir: {OOF_DIR}")


CELL S0: NB10b SHARED HELPERS
  V3 manifest: /mnt/f/PROJECTS/masterthesis/data_stack/dataset/v3/parquet_manifest.json
  Parquets registered: 36
  load_tier_parquets: 3 tiers, 63243 rows
  Cities: 21
  Points: 63243 (damaged=8247, undamaged=54996)
  EXPERIMENT_ID: 20260507_234413_32b992
  ResultRegistry: /content/drive_f/masterthesis/results/registry (run_id=20260507_234413)
  Helpers: load_v3(), prepare_Xy_from(), evaluate_groupkfold(), build_uisem()
  Output:  /content/drive_f/masterthesis/results/nb10b_v3
  OOF dir: /content/drive_f/masterthesis/results/nb10b_v3/oof_predictions


# CELL S0b: BUILDING-LEVEL AGGREGATION HELPER

Mirrors NB10a's `evaluate_3x3_max` protocol but uses the V3 `bda_points`
column `building_id` (Overture footprint IDs assigned at sampling time
in NB05bV3) as the grouping key, instead of NB10a's UTM 30 m grid binning
fallback.

This is the missing step that brings B1 / B2 / B3a in line with each paper's
actual reported unit:

- Ballinger 2024 PWTT: paper reports per-pixel detection but groups to
  building footprints via OSM mask before counting TP/FP. Our V3 PWTT
  produces per-pixel t-statistics; aggregating with max(|t|) per
  `building_id` reproduces the paper protocol.
- Scher 2025 CCD: paper reports per-pixel binary persistence flags then
  reduces to settlements / buildings via GHS-BUILT-S aggregation. Our V3
  CCD produces per-pixel `count_damage_months` and `binary_persistence`;
  aggregating with max (= OR for binary) per `building_id` reproduces the
  paper's reporting unit.
- Aimaiti 2022 log-ratio: paper applies the OSM-footprint mask **after**
  per-pixel thresholding, then counts a building as damaged if **any
  pixel** inside its footprint passes threshold. max-vote per
  `building_id` reproduces this exactly.

Points without a `building_id` (UNOSAT positives that did not snap to an
Overture footprint) are dropped from the building-level scores. The
per-point AUC numbers from B1 / B2 / B3a remain in the registry as the
strict per-pixel baseline; the new `_bldg` entries are the paper-protocol
numbers.


In [3]:
# @title CELL S0b: BUILDING-LEVEL AGGREGATION HELPER
print("=" * 70)
print("CELL S0b: BUILDING-LEVEL AGGREGATION HELPER")
print("=" * 70)


def aggregate_to_building(point_ids, cities, y_true, scores,
                          df_points_meta=None, drop_orphans=True):
    """Aggregate per-pixel predictions to per-building footprint via max-vote.

    Mirrors NB10a evaluate_3x3_max but uses the V3 bda_points building_id
    column (Overture footprint id) as the aggregation key. Points without a
    building_id (e.g. UNOSAT positives that did not snap to an Overture
    footprint) are dropped if drop_orphans=True.

    Parameters:
      point_ids       array-like of V3 point IDs
      cities          array-like of city names (parallel to point_ids)
      y_true          binary damage labels per point
      scores          predicted scores per point
      df_points_meta  optional override; defaults to global df_points which
                      must carry point_id, city, building_id columns
      drop_orphans    drop points with NA building_id (default True; matches
                      Aimaiti / Scher / Ballinger which all mask to footprints)

    Returns:
      DataFrame with one row per (city, building_id):
        city, building_id, n_points, y_true_max, score_max, score_mean
    """
    if df_points_meta is None:
        df_points_meta = df_points
    if 'building_id' not in df_points_meta.columns:
        raise RuntimeError(
            "df_points lacks 'building_id' column - cannot aggregate to "
            "building level. Re-load V3 bda_points with full schema.")
    df_pix = pd.DataFrame({
        'point_id': np.asarray(point_ids),
        'city': np.asarray(cities),
        'y_true': np.asarray(y_true).astype(int),
        'score': np.asarray(scores).astype(float),
    })
    pts_meta = df_points_meta[['point_id', 'city', 'building_id']].drop_duplicates(
        subset=['point_id', 'city'])
    df_pix = df_pix.merge(pts_meta, on=['point_id', 'city'], how='inner')
    n_total = len(df_pix)
    n_orphan = int(df_pix['building_id'].isna().sum())
    if drop_orphans:
        df_pix = df_pix.dropna(subset=['building_id'])
    if len(df_pix) == 0:
        print(f"    aggregate_to_building: no points with building_id "
              f"(n_orphan={n_orphan}/{n_total})")
        return None
    agg = df_pix.groupby(['city', 'building_id']).agg(
        n_points=('point_id', 'count'),
        y_true_max=('y_true', 'max'),
        score_max=('score', 'max'),
        score_mean=('score', 'mean'),
    ).reset_index()
    return agg


def evaluate_building_level(point_ids, cities, y_true, scores,
                              exp_name_pixel, exp_name_bldg, source_parquet,
                              classifier_name, cell_id, tags, note_extra=''):
    """Aggregate per-pixel predictions to building footprint, score, log,
    save OOF.

    Reports two AUCs (max-vote and mean-vote) and registers the better one
    as the headline. max-vote matches the paper protocol (Aimaiti: any
    pixel above threshold inside footprint -> damaged). mean-vote is the
    secondary diagnostic (less sensitive to per-pixel noise but smoother).
    """
    agg = aggregate_to_building(point_ids, cities, y_true, scores)
    if agg is None:
        print(f"  {exp_name_bldg}: aggregation produced no buildings")
        return None
    n_bldg = len(agg)
    n_dam = int((agg['y_true_max'] == 1).sum())
    n_und = int((agg['y_true_max'] == 0).sum())
    if agg['y_true_max'].nunique() < 2:
        print(f"  {exp_name_bldg}: only one class after aggregation "
              f"({n_dam}/{n_und})")
        return None
    auc_max = roc_auc_score(agg['y_true_max'].values, agg['score_max'].values)
    auc_mean = roc_auc_score(agg['y_true_max'].values, agg['score_mean'].values)
    auc_best = max(auc_max, auc_mean)
    direction = 'max' if auc_max >= auc_mean else 'mean'
    best_score = (agg['score_max'].values
                  if auc_max >= auc_mean else agg['score_mean'].values)
    avg_pix_per_bldg = float(agg['n_points'].mean())
    print(f"  {exp_name_bldg:<45s}: n_bldg={n_bldg:>6,} (dam={n_dam}/und={n_und})  "
          f"avg_pix/bldg={avg_pix_per_bldg:.1f}  "
          f"AUC(max)={auc_max:.3f}  AUC(mean)={auc_mean:.3f}  "
          f"best={auc_best:.3f} ({direction})")
    EXPERIMENT_LOG[exp_name_bldg] = {
        'auc': auc_best, 'f1': np.nan, 'n_features': 1,
        'method': 'unsupervised_aggregated', 'direction': direction,
        'aggregation': 'max_vote_building', 'n_buildings': n_bldg,
        'avg_pixels_per_building': avg_pix_per_bldg,
        'derived_from': exp_name_pixel,
    }
    registry.log_experiment(
        cell_id=cell_id, experiment_name=exp_name_bldg,
        parquet_name=source_parquet, tier_selection=_tiers,
        classifier_name=classifier_name,
        feature_set_name=f'{classifier_name}_bldg_{direction}',
        feature_cols=[exp_name_pixel],
        cv_method='unsupervised', n_folds=0, imputation='none',
        y_true=agg['y_true_max'].values, y_proba=best_score,
        groups=agg['city'].values,
        note=(f'building-level {direction}-vote aggregation of '
              f'{exp_name_pixel}{note_extra}'),
        tags=list(tags) + ['building_level', f'{direction}_vote'],
    )
    save_oof_unsupervised(
        point_ids=agg['building_id'].values,
        cities=agg['city'].values,
        y_true=agg['y_true_max'].values,
        scores=best_score,
        model_id=exp_name_bldg,
        variant_id=(f'unsupervised;aggregation=building_{direction}_vote;'
                    f'derived_from={exp_name_pixel}'),
    )
    return agg


# Verify df_points carries building_id before any B-cell is run
if 'building_id' not in df_points.columns:
    print(f"  WARNING: df_points missing 'building_id' column. "
          f"Available cols: {sorted(df_points.columns)}")
    print(f"  Building-level aggregation will fail when called.")
else:
    n_with = int(df_points['building_id'].notna().sum())
    n_without = int(df_points['building_id'].isna().sum())
    n_unique = int(df_points['building_id'].nunique())
    avg = (n_with / n_unique) if n_unique > 0 else 0.0
    print(f"  df_points.building_id: {n_with:,} points with footprint, "
          f"{n_without:,} orphans, {n_unique:,} unique buildings "
          f"(avg {avg:.1f} points/building)")
print("  Helpers: aggregate_to_building(), evaluate_building_level()")


CELL S0b: BUILDING-LEVEL AGGREGATION HELPER
  df_points.building_id: 63,243 points with footprint, 0 orphans, 41,326 unique buildings (avg 1.5 points/building)
  Helpers: aggregate_to_building(), evaluate_building_level()


# CELL S0c: OPTUNA HELPERS (mirrors NB10a H10a)

Imports Optuna and defines:

- `SEARCH_SPACES` — dict of `{classifier_name: (suggest_fn, classifier_class)}`
  for RF, ExtraTrees, GBM (sklearn), XGBoost, LightGBM. Identical bounds to
  NB10a H10a so tuning budgets transfer directly.
- `run_optuna_groupkfold()` — GroupKFold-only Optuna runner (no Dietrich
  4/14 split here because all 21 UNOSAT cities are used). Same TPESampler /
  MedianPruner as NB10a.

Trial budgets follow NB10a's convention:

- `OPTUNA_TRIALS_FAITHFUL = 300` for the headline tuned faithful number
- `OPTUNA_TRIALS_BEST_OTHER = 200` for ceiling-section runs

These are imported here so this notebook does not depend on NB10a being
loaded in the same kernel session.


In [4]:
# @title CELL S0c: OPTUNA HELPERS (mirror NB10a H10a)
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier,
                              GradientBoostingClassifier)

print("=" * 70)
print("CELL S0c: NB10b OPTUNA HELPERS")
print("=" * 70)

# Trial budgets (mirror NB10a CONFIG)
OPTUNA_TRIALS_FAITHFUL  = OPTUNA_TRIALS_DEFAULT   # alias from CONFIG (200)
OPTUNA_TRIALS_BEST_OTHER = OPTUNA_TRIALS_DEFAULT  # alias from CONFIG (200)


def _suggest_rf(trial):
    return {
        'n_estimators':      trial.suggest_int('n_estimators', 100, 1000, step=50),
        'max_depth':         trial.suggest_int('max_depth', 5, 50),
        'min_samples_leaf':  trial.suggest_int('min_samples_leaf', 1, 20),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'max_features':      trial.suggest_categorical('max_features',
                                ['sqrt', 'log2', 0.3, 0.5, 0.7]),
        'class_weight':      trial.suggest_categorical('class_weight',
                                ['balanced', None]),
        'random_state':      RANDOM_STATE, 'n_jobs': -1,
    }

def _suggest_extratrees(trial):
    return {
        'n_estimators':      trial.suggest_int('n_estimators', 100, 1000, step=50),
        'max_depth':         trial.suggest_int('max_depth', 5, 50),
        'min_samples_leaf':  trial.suggest_int('min_samples_leaf', 1, 20),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'max_features':      trial.suggest_categorical('max_features',
                                ['sqrt', 'log2', 0.3, 0.5, 0.7]),
        'class_weight':      trial.suggest_categorical('class_weight',
                                ['balanced', None]),
        'random_state':      RANDOM_STATE, 'n_jobs': -1,
    }

def _suggest_xgboost(trial):
    return {
        'n_estimators':      trial.suggest_int('n_estimators', 100, 2000, step=50),
        'max_depth':         trial.suggest_int('max_depth', 3, 15),
        'learning_rate':     trial.suggest_float('learning_rate', 1e-3, 0.3, log=True),
        'subsample':         trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight':  trial.suggest_int('min_child_weight', 1, 20),
        'reg_alpha':         trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'scale_pos_weight':  trial.suggest_float('scale_pos_weight', 1.0, 100.0, log=True),
        'random_state':      RANDOM_STATE, 'n_jobs': -1, 'verbosity': 0,
        'eval_metric':       'logloss', 'use_label_encoder': False,
    }

def _suggest_lightgbm(trial):
    return {
        'n_estimators':      trial.suggest_int('n_estimators', 100, 2000, step=50),
        'num_leaves':        trial.suggest_int('num_leaves', 15, 300),
        'learning_rate':     trial.suggest_float('learning_rate', 1e-3, 0.3, log=True),
        'feature_fraction':  trial.suggest_float('feature_fraction', 0.5, 1.0),
        'bagging_fraction':  trial.suggest_float('bagging_fraction', 0.5, 1.0),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        'reg_alpha':         trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'is_unbalance':      trial.suggest_categorical('is_unbalance', [True, False]),
        'random_state':      RANDOM_STATE, 'n_jobs': -1, 'verbose': -1,
    }

def _suggest_gbm(trial):
    return {
        'n_estimators':     trial.suggest_int('n_estimators', 100, 1000, step=50),
        'max_depth':        trial.suggest_int('max_depth', 3, 10),
        'learning_rate':    trial.suggest_float('learning_rate', 1e-3, 0.3, log=True),
        'subsample':        trial.suggest_float('subsample', 0.5, 1.0),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 20),
        'random_state':     RANDOM_STATE,
    }

SEARCH_SPACES = {
    'RF':         (_suggest_rf,        RandomForestClassifier),
    'ExtraTrees': (_suggest_extratrees, ExtraTreesClassifier),
    'GBM':        (_suggest_gbm,       GradientBoostingClassifier),
}

try:
    from xgboost import XGBClassifier
    SEARCH_SPACES['XGBoost'] = (_suggest_xgboost, XGBClassifier)
except ImportError:
    print("  XGBoost not available -- search space skipped")

try:
    from lightgbm import LGBMClassifier as _LGBMClassifier
    SEARCH_SPACES['LightGBM'] = (_suggest_lightgbm, _LGBMClassifier)
except ImportError:
    print("  LightGBM not available -- search space skipped")


def run_optuna_groupkfold(classifier_name, X, y, groups, n_trials, study_name,
                           n_folds=N_FOLDS, point_ids=None, needs_scaling=False,
                           native_nan=False):
    """GroupKFold Optuna runner. Mirrors NB10a run_optuna(cv_method=
    'groupkfold') but lives in this notebook for portability.

    native_nan=True: do not impute NaNs; relies on the classifier's own
    NaN handling (LightGBM and XGBoost both support this). For sklearn
    estimators the SimpleImputer in evaluate_groupkfold still runs.

    Returns dict {'best_params', 'best_auc', 'best_result', 'study'}.
    """
    if classifier_name not in SEARCH_SPACES:
        raise ValueError(
            f"Unknown classifier: {classifier_name}. "
            f"Available: {list(SEARCH_SPACES.keys())}")
    suggest_fn, clf_class = SEARCH_SPACES[classifier_name]

    def objective(trial):
        params = suggest_fn(trial)
        clf = clf_class(**params)
        res = evaluate_groupkfold(
            clf, X, y, groups,
            label=f"{study_name}_trial{trial.number}",
            n_folds=n_folds, point_ids=point_ids,
        )
        if res is None:
            return 0.0
        EXPERIMENT_LOG.pop(f"{study_name}_trial{trial.number}", None)
        return res['auc']

    sampler = TPESampler(seed=RANDOM_STATE)
    pruner = MedianPruner(n_startup_trials=10, n_warmup_steps=0)
    study = optuna.create_study(direction='maximize', sampler=sampler,
                                 pruner=pruner, study_name=study_name)
    print(f"  Optuna {classifier_name} ({n_trials} trials, study={study_name})")
    t0 = time.time()
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    elapsed = time.time() - t0
    print(f"    best AUC = {study.best_value:.4f}  ({elapsed/60:.1f} min)")
    print(f"    best params: {study.best_params}")

    best_params = dict(study.best_params)
    suggest_dummy = suggest_fn(optuna.trial.FixedTrial(study.best_params))
    for k in ('random_state', 'n_jobs', 'verbosity', 'verbose', 'eval_metric',
              'use_label_encoder'):
        if k in suggest_dummy and k not in best_params:
            best_params[k] = suggest_dummy[k]
    best_clf = clf_class(**best_params)
    best_res = evaluate_groupkfold(
        best_clf, X, y, groups,
        label=study_name + '_BEST',
        n_folds=n_folds, point_ids=point_ids,
    )
    return {
        'best_params': best_params, 'best_auc': study.best_value,
        'best_result': best_res, 'study': study,
        'classifier_name': classifier_name, 'n_trials': n_trials,
        'elapsed_min': elapsed / 60,
    }


print(f"  Search spaces: {list(SEARCH_SPACES.keys())}")
print(f"  Trial budgets: faithful={OPTUNA_TRIALS_FAITHFUL}, "
      f"best-other={OPTUNA_TRIALS_BEST_OTHER}")


CELL S0c: NB10b OPTUNA HELPERS
  Search spaces: ['RF', 'ExtraTrees', 'GBM', 'XGBoost', 'LightGBM']
  Trial budgets: faithful=200, best-other=200


# Paper 1: Ballinger 2024 -- Pixel-Wise t-Test (PWTT)

> Ballinger, O. (2024). Open access battle damage detection via pixel-wise
> t-test on Sentinel-1 imagery. arXiv:2405.06323.
> Reported AUC = 0.757, F1 = 0.684 in Dietrich et al. 2025 Table 2
> (with PWTT calibrated against UNOSAT for Ukraine).

## Method (paper)

For each pixel, run Welch's two-sample t-test on the Sentinel-1 amplitude
time series, comparing a pre-event window against a post-event window.
Damaged pixels are those whose |t| exceeds a threshold calibrated against
UNOSAT damage points. The original implementation is in Google Earth Engine
and operates per-pixel per-orbit on Sentinel-1 GRD VV amplitude.

## In scope for NB10b (faithful)

- **Per-pixel sampling.** V3 `bda_scene_card_t{tier}.parquet` -- one row per
  (point_id, city, date), columns `s1__vv` and `s1__vh` are single-pixel
  values sampled at the UNOSAT damage point or sampled negative point.
- **Welch t-test on full time series.** Vectorised per (point_id, polarisation)
  across all pre-battle dates vs all crossbattle+postbattle dates with
  >=2 observations per side. Welch's formula:
  t = (mean_post - mean_pre) / sqrt(var_pre/n_pre + var_post/n_post)
- **Period split via `period_label`.** Set in NB05bV3 from each city's
  battle_start / battle_stop in `data_stack_manifest.json`. Matches Dietrich's
  tunosat-cutoff convention (post = crossbattle + postbattle).
- **VV, VH, and VV+VH combined.** Paper headline is VV; we report all three.
- **AUC with direction selection.** Both |t| and -t are evaluated; the
  better of the two is reported per polarisation. Threshold-free.

## Out of scope for NB10b (documented divergences)

- **Calibrated decision threshold.** Paper calibrates a threshold against
  UNOSAT directly. We use AUC; F1/precision/recall reported here use a
  Youden's J cutoff on the same data, which differs from the paper's
  protocol (paper holds out a calibration set). Not equivalent to paper F1.
- **Per-orbit asc/desc separation.** CARD is single-orbit per city by data
  stack convention (one orbit chosen per city in NB02). Paper averages
  across orbits at each location.
- **3-month rolling assessment windows.** Paper uses fixed pre and rolling
  3-month post windows (T1..T12) in Dietrich's terminology. We collapse all
  crossbattle + postbattle dates into a single "post" group. 3-month windowing
  would require regrouping `scene_card` by month and is deferred.
- **GEE deployment context.** Paper runs in Google Earth Engine; we run
  locally on V3 parquets. Numerical equivalence assumed since both consume
  Sentinel-1 GRD amplitudes after standard preprocessing.

## What changed vs NB09b B1

NB09b's "EXACT mode" looked for column names containing both `s1__{pol}` and
`mean` (a V2 footprint convention). On V3 the column is bare `s1__vv` (no
`__mean` suffix), so the candidate list was empty and NB09b silently fell
through to an "APPROX" path that computed a single-scene pseudo-t-statistic
from `prepost_single_card`. That path is not Welch's t-test on a time series
and should not be reported as PWTT. NB10b uses the V3 bare column names
directly and removes the APPROX fallback.

In [5]:
# @title CELL B1: PWTT FAITHFUL (Ballinger 2024) -- V3 per-point
print("=" * 70)
print("CELL B1: PWTT FAITHFUL (Ballinger 2024) -- V3 scene_card")
print("=" * 70)

def vectorized_welch_ttest_v3(scene_df, val_col, pol_label):
    '''Welch's two-sample t-test per point on a V3 long scene_card frame.

    Pre = period_label == 'prebattle'.
    Post = period_label in ('crossbattle', 'postbattle').
    Requires >= 2 observations per side per (city, point_id).
    Returns DataFrame with one row per (city, point_id) and one column
    `pwtt_t_{pol_label}`.
    '''
    tmp = scene_df[['city', 'point_id', 'period_label', val_col]].copy()
    tmp = tmp.dropna(subset=[val_col])
    tmp['period_group'] = 'drop'
    tmp.loc[tmp['period_label'] == 'prebattle', 'period_group'] = 'pre'
    tmp.loc[tmp['period_label'].isin(['crossbattle', 'postbattle']), 'period_group'] = 'post'
    tmp = tmp[tmp['period_group'] != 'drop']
    if len(tmp) == 0:
        return None
    agg = tmp.groupby(['city', 'point_id', 'period_group'])[val_col].agg(
        ['mean', 'std', 'count']).reset_index()
    pre = agg[agg['period_group'] == 'pre'].set_index(['city', 'point_id'])
    post = agg[agg['period_group'] == 'post'].set_index(['city', 'point_id'])
    joined = pre.join(post, lsuffix='_pre', rsuffix='_post', how='inner')
    joined = joined[(joined['count_pre'] >= 2) & (joined['count_post'] >= 2)]
    if len(joined) == 0:
        return None
    var_pre = joined['std_pre'] ** 2
    var_post = joined['std_post'] ** 2
    n_pre = joined['count_pre']
    n_post = joined['count_post']
    denom = np.sqrt(var_pre / n_pre + var_post / n_post + 1e-30)
    joined[f'pwtt_t_{pol_label}'] = (joined['mean_post'] - joined['mean_pre']) / denom
    out = joined[[f'pwtt_t_{pol_label}', 'count_pre', 'count_post']].reset_index()
    out = out.rename(columns={'count_pre': f'n_pre_{pol_label}',
                              'count_post': f'n_post_{pol_label}'})
    return out


def eval_pwtt_v3(scores_df, t_col, exp_name, pol_label, source_parquet):
    '''Score, log to registry, save OOF.'''
    if scores_df is None or len(scores_df) < 50:
        print(f"  {exp_name}: insufficient data ({0 if scores_df is None else len(scores_df)})")
        return None
    y = scores_df[TARGET_COL].values
    if len(np.unique(y)) < 2:
        print(f"  {exp_name}: only one class")
        return None
    scores_abs = np.abs(scores_df[t_col].values)
    scores_neg = -scores_df[t_col].values
    auc_abs = roc_auc_score(y, scores_abs)
    auc_neg = roc_auc_score(y, scores_neg)
    auc_best = max(auc_abs, auc_neg)
    direction = "|t|" if auc_abs >= auc_neg else "-t"
    best_scores = scores_abs if auc_abs >= auc_neg else scores_neg
    n_dam = int((y == 1).sum()); n_und = int((y == 0).sum())
    print(f"  {exp_name:<35s} ({pol_label}): n={len(y):>6d} (dam={n_dam}/und={n_und})  "
          f"AUC(|t|)={auc_abs:.3f}  AUC(-t)={auc_neg:.3f}  best={auc_best:.3f} ({direction})")
    EXPERIMENT_LOG[exp_name] = {'auc': auc_best, 'f1': np.nan, 'n_features': 1,
                                 'method': 'unsupervised', 'direction': direction}
    ALL_SCORES[f'pwtt_{pol_label}'] = pd.DataFrame({
        'point_id': scores_df['point_id'].values,
        'city': scores_df['city'].values,
        f'score_pwtt_{pol_label}': best_scores,
    })
    registry.log_experiment(
        cell_id='cell_b1', experiment_name=exp_name,
        parquet_name=source_parquet, tier_selection=_tiers,
        classifier_name='unsupervised_ttest',
        feature_set_name=f'pwtt_{pol_label}', feature_cols=[t_col],
        cv_method='unsupervised', n_folds=0, imputation='none',
        y_true=y, y_proba=best_scores, groups=scores_df['city'].values,
        note=f'Ballinger 2024 PWTT {pol_label} on V3 scene_card (faithful), '
             f'paper AUC={BALLINGER_PAPER["auc"]}',
        tags=['unsupervised', 'pwtt', 'ballinger', 'faithful', pol_label],
    )
    save_oof_unsupervised(
        point_ids=scores_df['point_id'].values,
        cities=scores_df['city'].values,
        y_true=y, scores=best_scores, model_id=exp_name,
        variant_id=f'unsupervised_pwtt;pol={pol_label};direction={direction};'
                   f'mode=v3_pixel_timeseries',
    )
    return auc_best


# --- load V3 scene_card ----------------------------------------------
df_sc = load_v3('scene_card')

if 'period_label' not in df_sc.columns:
    raise RuntimeError(
        "scene_card V3 missing 'period_label' column. Faithful PWTT requires "
        "per-date period labelling from NB05bV3. Re-run NB05bV3 cell 6 "
        "with FR_SCENE_CARD=True to regenerate.")

# verify the V3 bare column names are present
for pol in ('vv', 'vh'):
    expected = f's1__{pol}'
    if expected not in df_sc.columns:
        raise RuntimeError(
            f"scene_card V3 missing column '{expected}'. Available SAR cols: "
            f"{[c for c in df_sc.columns if 's1__' in c]}")

# --- per (point, polarisation) Welch t-test --------------------------
pwtt_vv_scores = None
pwtt_vh_scores = None

for pol in ('vv', 'vh'):
    vcol = f's1__{pol}'
    print(f"\n  --- Polarisation {pol.upper()} (column = {vcol}) ---")
    res_ttest = vectorized_welch_ttest_v3(df_sc, vcol, pol)
    if res_ttest is None:
        print(f"    no points with >=2 pre AND >=2 post observations")
        continue
    merged = res_ttest.merge(df_points[['point_id', 'city', TARGET_COL]].drop_duplicates(),
                              on=['point_id', 'city'], how='inner')
    print(f"    points with valid t-statistic: {len(merged):,} "
          f"(mean n_pre={merged[f'n_pre_{pol}'].mean():.1f}, "
          f"mean n_post={merged[f'n_post_{pol}'].mean():.1f})")
    if pol == 'vv':
        pwtt_vv_scores = merged
        eval_pwtt_v3(merged, 'pwtt_t_vv', 'B1_PWTT_VV_faithful', 'vv',
                     'bda_scene_card_v3')
    else:
        pwtt_vh_scores = merged
        eval_pwtt_v3(merged, 'pwtt_t_vh', 'B1_PWTT_VH_faithful', 'vh',
                     'bda_scene_card_v3')

# --- VV + VH combined ------------------------------------------------
if pwtt_vv_scores is not None and pwtt_vh_scores is not None:
    print(f"\n  --- VV+VH combined ---")
    merged = pwtt_vv_scores[['point_id', 'city', TARGET_COL, 'pwtt_t_vv']].merge(
        pwtt_vh_scores[['point_id', 'city', 'pwtt_t_vh']],
        on=['point_id', 'city'], how='inner')
    if len(merged) > 50:
        combined_t = np.abs(merged['pwtt_t_vv'].values) + np.abs(merged['pwtt_t_vh'].values)
        y = merged[TARGET_COL].values
        if len(np.unique(y)) >= 2:
            auc_comb = roc_auc_score(y, combined_t)
            print(f"    PWTT VV+VH combined: AUC={auc_comb:.3f}  ({len(merged):,} points)")
            EXPERIMENT_LOG['B1_PWTT_VV+VH_faithful'] = {
                'auc': auc_comb, 'f1': np.nan, 'n_features': 2,
                'method': 'unsupervised', 'direction': '|t_vv|+|t_vh|',
            }
            registry.log_experiment(
                cell_id='cell_b1', experiment_name='B1_PWTT_VV+VH_faithful',
                parquet_name='bda_scene_card_v3', tier_selection=_tiers,
                classifier_name='unsupervised_ttest',
                feature_set_name='pwtt_vv+vh', feature_cols=['pwtt_t_vv', 'pwtt_t_vh'],
                cv_method='unsupervised', n_folds=0, imputation='none',
                y_true=y, y_proba=combined_t, groups=merged['city'].values,
                note='Ballinger 2024 PWTT VV+VH combined |t| sum on V3 scene_card (faithful)',
                tags=['unsupervised', 'pwtt', 'ballinger', 'faithful', 'combined'],
            )
            save_oof_unsupervised(
                point_ids=merged['point_id'].values,
                cities=merged['city'].values,
                y_true=y, scores=combined_t, model_id='B1_PWTT_VV+VH_faithful',
                variant_id='unsupervised_pwtt;pol=vv+vh;direction=|t_vv|+|t_vh|;'
                           'mode=v3_pixel_timeseries',
            )

print(f"\n  Ballinger paper reference: AUC={BALLINGER_PAPER['auc']}, F1={BALLINGER_PAPER['f1']}")

del df_sc
gc.collect()


CELL B1: PWTT FAITHFUL (Ballinger 2024) -- V3 scene_card
  load_v3('scene_card'): 834,134 rows, 8 cols, 21 cities

  --- Polarisation VV (column = s1__vv) ---
    points with valid t-statistic: 62,151 (mean n_pre=5.0, mean n_post=8.3)
  B1_PWTT_VV_faithful                 (vv): n= 62151 (dam=8013/und=54138)  AUC(|t|)=0.526  AUC(-t)=0.485  best=0.526 (|t|)
  REG: B1_PWTT_VV_faithful                           AUC=0.5260 F1=0.2245 n=62151 feat=1 cities=20 [NB10b_v3/cell_b1]
    saved oof -> oof_B1_PWTT_VV_faithful__20260507_234413_32b992.parquet  TP=4243 FN=3770 FP=26833 TN=27305

  --- Polarisation VH (column = s1__vh) ---
    points with valid t-statistic: 62,151 (mean n_pre=5.0, mean n_post=8.3)
  B1_PWTT_VH_faithful                 (vh): n= 62151 (dam=8013/und=54138)  AUC(|t|)=0.531  AUC(-t)=0.512  best=0.531 (|t|)
  REG: B1_PWTT_VH_faithful                           AUC=0.5312 F1=0.2269 n=62151 feat=1 cities=20 [NB10b_v3/cell_b1]
    saved oof -> oof_B1_PWTT_VH_faithful__20260507_234

0

In [6]:
# @title CELL B1_bldg: PWTT BUILDING-LEVEL AGGREGATION (max-vote)
print("=" * 70)
print("CELL B1_bldg: PWTT BUILDING-LEVEL AGGREGATION (max-vote)")
print("=" * 70)
print("  Aggregating per-pixel PWTT predictions to per-building footprint")
print("  via Overture building_id (matches Ballinger paper protocol of")
print("  scoring buildings by max |t| over pixels in OSM footprint).")
print()


def _direction_to_score(scores_df, t_col, exp_name):
    """Recover the AUC-best direction recorded by eval_pwtt_v3 and
    return the corresponding score column."""
    info = EXPERIMENT_LOG.get(exp_name, {})
    direction = info.get('direction', '|t|')
    raw = scores_df[t_col].values
    if direction == '|t|':
        return np.abs(raw), direction
    if direction == '-t':
        return -raw, direction
    return raw, direction


# --- VV --------------------------------------------------------------
if pwtt_vv_scores is not None:
    score_vv, dir_vv = _direction_to_score(
        pwtt_vv_scores, 'pwtt_t_vv', 'B1_PWTT_VV_faithful')
    print(f"  --- VV (per-pixel direction = {dir_vv}) ---")
    evaluate_building_level(
        point_ids=pwtt_vv_scores['point_id'].values,
        cities=pwtt_vv_scores['city'].values,
        y_true=pwtt_vv_scores[TARGET_COL].values,
        scores=score_vv,
        exp_name_pixel='B1_PWTT_VV_faithful',
        exp_name_bldg='B1_PWTT_VV_faithful_bldg',
        source_parquet='bda_scene_card_v3',
        classifier_name='unsupervised_ttest',
        cell_id='cell_b1_bldg',
        tags=['unsupervised', 'pwtt', 'ballinger', 'faithful', 'vv'],
        note_extra=f' (paper AUC={BALLINGER_PAPER["auc"]})',
    )

# --- VH --------------------------------------------------------------
if pwtt_vh_scores is not None:
    score_vh, dir_vh = _direction_to_score(
        pwtt_vh_scores, 'pwtt_t_vh', 'B1_PWTT_VH_faithful')
    print(f"  --- VH (per-pixel direction = {dir_vh}) ---")
    evaluate_building_level(
        point_ids=pwtt_vh_scores['point_id'].values,
        cities=pwtt_vh_scores['city'].values,
        y_true=pwtt_vh_scores[TARGET_COL].values,
        scores=score_vh,
        exp_name_pixel='B1_PWTT_VH_faithful',
        exp_name_bldg='B1_PWTT_VH_faithful_bldg',
        source_parquet='bda_scene_card_v3',
        classifier_name='unsupervised_ttest',
        cell_id='cell_b1_bldg',
        tags=['unsupervised', 'pwtt', 'ballinger', 'faithful', 'vh'],
    )

# --- VV+VH combined --------------------------------------------------
if pwtt_vv_scores is not None and pwtt_vh_scores is not None:
    print(f"  --- VV+VH combined (|t_vv| + |t_vh|) ---")
    merged = pwtt_vv_scores[['point_id', 'city', TARGET_COL, 'pwtt_t_vv']].merge(
        pwtt_vh_scores[['point_id', 'city', 'pwtt_t_vh']],
        on=['point_id', 'city'], how='inner')
    if len(merged) > 50:
        combined = (np.abs(merged['pwtt_t_vv'].values)
                    + np.abs(merged['pwtt_t_vh'].values))
        evaluate_building_level(
            point_ids=merged['point_id'].values,
            cities=merged['city'].values,
            y_true=merged[TARGET_COL].values,
            scores=combined,
            exp_name_pixel='B1_PWTT_VV+VH_faithful',
            exp_name_bldg='B1_PWTT_VV+VH_faithful_bldg',
            source_parquet='bda_scene_card_v3',
            classifier_name='unsupervised_ttest',
            cell_id='cell_b1_bldg',
            tags=['unsupervised', 'pwtt', 'ballinger', 'faithful', 'combined'],
        )

print(f"\n  Ballinger paper reference: AUC={BALLINGER_PAPER['auc']}, "
      f"F1={BALLINGER_PAPER['f1']}")


CELL B1_bldg: PWTT BUILDING-LEVEL AGGREGATION (max-vote)
  Aggregating per-pixel PWTT predictions to per-building footprint
  via Overture building_id (matches Ballinger paper protocol of
  scoring buildings by max |t| over pixels in OSM footprint).

  --- VV (per-pixel direction = |t|) ---
  B1_PWTT_VV_faithful_bldg                     : n_bldg=56,562 (dam=6920/und=49642)  avg_pix/bldg=1.1  AUC(max)=0.539  AUC(mean)=0.533  best=0.539 (max)
  REG: B1_PWTT_VV_faithful_bldg                      AUC=0.5391 F1=0.2171 n=56562 feat=1 cities=20 [NB10b_v3/cell_b1_bldg]
    saved oof -> oof_B1_PWTT_VV_faithful_bldg__20260507_234413_32b992.parquet  TP=3786 FN=3134 FP=24496 TN=25146
  --- VH (per-pixel direction = |t|) ---
  B1_PWTT_VH_faithful_bldg                     : n_bldg=56,562 (dam=6920/und=49642)  avg_pix/bldg=1.1  AUC(max)=0.546  AUC(mean)=0.541  best=0.546 (max)
  REG: B1_PWTT_VH_faithful_bldg                      AUC=0.5461 F1=0.2205 n=56562 feat=1 cities=20 [NB10b_v3/cell_b1_bldg]
  

# Paper 2: Scher & Van Den Hoek 2025 -- Coherent Change Detection (CCD)

> Scher, C., & Van Den Hoek, J. (2025). Nationwide conflict damage mapping
> with interferometric synthetic aperture radar: A study of the 2022
> Russia-Ukraine conflict. Science of Remote Sensing 11, 100217.
> Reported on UNOSAT test partition: TPR = 59.1%, FPR = 1.14%, F1 = 0.68.

## Method (paper)

For each pixel:

1. **Pre-conflict baseline.** Coherence values formed during May 2020 - Dec
   2021 are reduced to a long-term mean `gamma_pre_mean` and standard
   deviation `gamma_pre_std`.
2. **Monthly maximum during the conflict.** Per pixel, take the max
   coherence value within each calendar month of the monitoring period
   (March 2022 - Oct 2023). One value per (pixel, month).
3. **Per-month damage criterion.** Both must hold simultaneously:
   - delta_gamma_t = gamma_t - gamma_pre_mean < k        (k = -0.05)
   - z_t = delta_gamma_t / gamma_pre_std < z_thresh      (z_thresh = -2)
4. **Validity mask.** A pixel is valid for monitoring iff
   z_k = (k - gamma_pre_mean) / gamma_pre_std < z_thresh.
   Pixels failing this mask are excluded entirely (not counted as TP/FP/FN/TN).
5. **Persistence.** Damage is assigned only if the per-month criterion
   holds for >= 3 consecutive calendar months.

Output is binary (damaged / undamaged). Reported numerically with TPR, FPR,
F1, CSI against UNOSAT comprehensive damage assessment data partitioned 50/50
into calibration and test halves.

## In scope for NB10b (faithful)

- **V3 `scene_coh` A3** at `bda_scene_coh_t{tier}.parquet`. Each row =
  (city, point_id, date1, date2, period_label, s1__coh_vv, s1__coh_vh,
  s1__coh_vv__zscore). The pair date `date2` is treated as the
  observation date.
- **Per-pixel pre baseline** = mean and std of `s1__coh_{vv|vh}` over rows
  where `period_label == 'prebattle'` per (city, point_id). Requires
  >= 2 prebattle observations.
- **Monthly max** = `groupby((city, point_id, year_month)).max()` of
  `s1__coh_{vv|vh}` over rows where `period_label in ('crossbattle',
  'postbattle')`. `year_month` derived from `date2` parsed as `%Y%m%d`.
- **Validity mask z_k < z_thresh** computed exactly per Eq. 5.
- **Per-month criterion** delta < k AND z < z_thresh exactly per Eq. 3.
- **Persistence >= N consecutive months.** Implemented vectorised: sort by
  year_month within (city, point_id), mark a "new run" at any damage row
  whose previous damage row is either non-damage or not the immediately
  preceding calendar month, then cumsum to assign run ids and take the
  max run length per pixel.
- **Defaults from paper.** k = -0.05, z_thresh = -2.0, persistence_months = 3.
- **Three score outputs per pixel** for downstream evaluation:
  - `count_damage_months` (continuous, for AUC)
  - `max_run_months` (continuous, for AUC)
  - `binary_persistence` (binary, for paper-protocol F1/precision/recall)
- **VV (headline), VH, VV+VH (sum count, OR persistence) reported.**

## Out of scope for NB10b (documented divergences)

- **Long-temporal-baseline interferograms.** Paper forms each conflict-period
  coherence pair against three fixed Jul-Aug 2021 reference scenes
  (long temporal baseline). Our V3 `scene_coh` carries 12-day natural-baseline
  pairs from the SLC pairs selected in NB02. Twelve-day coherence is
  numerically higher than long-baseline coherence over stable areas, so the
  empirical k that minimises false alarms in our pipeline may differ from
  the paper's k = -0.05. Faithful run uses paper's k regardless; Optuna
  cell will calibrate on training fold.
- **Single fixed nationwide reference window.** Paper uses May 2020 - Dec 2021
  globally. Our `period_label` is city-relative to `battle_start`, so the
  prebattle window length differs across cities (Bakhmut has years of
  prebattle data, Mariupol has only ~2 years).
- **GHS-BUILT-S validity mask.** Paper restricts monitoring to GHS-BUILT-S
  pixels >= 11 m^2 of built-up surface. Our V3 sample inventory consists of
  UNOSAT positives plus negatives sampled from Overture building centroids,
  so all sample points are by construction within or near built-up areas.
  No additional mask applied here.
- **Counterfactual period for FPR.** Paper computes FPR from a 2019 period
  before any damage took place. We have no counterfactual; "FPR" in our
  reporting is computed against sampled negatives in the same monitoring
  period and is not directly comparable to the paper's 1.14%.
- **40 m projected pixel spacing.** Paper estimates coherence with a 10 x 2
  multi-look in radar geometry, yielding ~40 m output pixels. Our pipeline
  produces 10 m coherence rasters and we sample at the point's native
  10 m grid. Out of scope; would require regenerating coherence in NB02.
- **Paper's open-pit-mining mask, frontline analysis, settlement aggregation.**
  Paper applies these as post-processing for spatial-temporal analysis;
  not relevant to per-pixel detection metrics reported here.

## What changed vs NB09b B2

NB09b B2 had two paths:

1. **EXACT path** read `scene_coh` (A3) but searched for coherence columns
   matching both 'coh' and 'mean' (a V2 zonal-stats convention). On V3 the
   column is bare `s1__coh_vv` so this path either grabbed the wrong column
   (`s1__coh_vv__zscore` if the substring search reached it) or fell through
   entirely. When it did execute, it computed `pre_mean - post_mean` over the
   collapsed pre / post split with no monthly aggregation, no validity mask,
   no z-score check, no persistence. That is not Scher's CCD.
2. **APPROX path** read `coh_drop` A14 -- already aggregated across the
   entire post period. Cannot do per-month tracking or persistence
   from those columns. The user's note "fusion drop is aggregation over
   all battle months directly done" applies to this path: it implements a
   simpler "any drop ever" detector, useful as a comparison run but not
   faithful to Scher.

NB10b B2 reads V3 `scene_coh` A3 directly, uses bare `s1__coh_{pol}`
columns, and implements all four paper criteria (monthly max, delta < k,
z < z_thresh, validity mask, persistence). The NB09b APPROX path remains
in the registry as a separate model (`B2_CCD_approx_*`) for the easy-vs-faithful
comparison the user mentioned.

## A note on the precomputed `s1__coh_vv__zscore` column

NB03e R1 already produces a per-date z-score raster `(coh_t - gamma_pre_mean)
/ gamma_pre_std` at the city level, and NB05bV3 samples it at the point as
`s1__coh_vv__zscore`. This is mathematically equivalent to the per-date
z-score the paper computes (only VV; no VH zscore raster is produced).
However the paper requires per-MONTH z-scores derived from the monthly max
coh, and aggregating an already-z-scored per-date series via min would not
give exactly the same result if the month spans multiple observations
(min-of-z is not z-of-max in general). NB10b therefore computes z from
scratch using the raw `s1__coh_{pol}` column for both polarisations, which
is the paper's exact protocol. The precomputed z-score column is unused
here.

In [7]:
# @title CELL B2: CCD FAITHFUL (Scher & Van Den Hoek 2025) -- V3 per-point
print("=" * 70)
print("CELL B2: CCD FAITHFUL (Scher 2025) -- V3 scene_coh")
print("=" * 70)


def compute_pre_baseline(scene_df, val_col):
    '''Per-pixel mean and std of pre-conflict coherence.

    Returns DataFrame with one row per (city, point_id) carrying
    coh_pre_mean, coh_pre_std, n_pre, coh_pre_std_safe (clipped at 1e-6
    to avoid division by zero in z-scoring). Drops pixels with < 2
    prebattle observations.
    '''
    pre = scene_df[scene_df['period_label'] == 'prebattle'][
        ['city', 'point_id', val_col]].copy()
    pre = pre.dropna(subset=[val_col])
    stats = pre.groupby(['city', 'point_id'])[val_col].agg(['mean', 'std', 'count']).reset_index()
    stats.columns = ['city', 'point_id', 'coh_pre_mean', 'coh_pre_std', 'n_pre']
    stats = stats[stats['n_pre'] >= 2].copy()
    stats['coh_pre_std_safe'] = stats['coh_pre_std'].fillna(1e-6).clip(lower=1e-6)
    return stats


def compute_monthly_max_post(scene_df, val_col):
    '''Per-pixel monthly max of post-conflict coherence.

    Post = period_label in ('crossbattle', 'postbattle'). Date column 'date'
    in V3 scene_coh equals the second date of the InSAR pair (the
    monitoring date). Bucketed to year_month YYYY-MM.

    Returns DataFrame with columns (city, point_id, year_month, coh_t).
    '''
    post = scene_df[scene_df['period_label'].isin(['crossbattle', 'postbattle'])][
        ['city', 'point_id', 'date', val_col]].copy()
    post = post.dropna(subset=[val_col])
    post['date_dt'] = pd.to_datetime(post['date'], format='%Y%m%d', errors='coerce')
    post = post[post['date_dt'].notna()]
    post['year_month'] = post['date_dt'].dt.to_period('M').astype(str)
    monthly = post.groupby(['city', 'point_id', 'year_month'])[val_col].max().reset_index()
    monthly = monthly.rename(columns={val_col: 'coh_t'})
    return monthly


def apply_ccd_paper(pre_stats, monthly, k=-0.05, z_thresh=-2.0,
                    persistence_months=3):
    '''Apply Scher 2025 CCD: validity + per-month criteria + persistence.

    Returns:
        out: DataFrame one row per valid pixel with columns
            city, point_id, count_damage_months, max_run_months,
            n_post_months, binary_persistence
        monthly_full: the per-month frame (joined with baseline) used to
            compute the above. Useful for diagnostics.
    '''
    pre_stats = pre_stats.copy()
    pre_stats['z_k'] = (k - pre_stats['coh_pre_mean']) / pre_stats['coh_pre_std_safe']
    pre_stats['valid_for_monitoring'] = pre_stats['z_k'] < z_thresh

    # join monthly observations with per-pixel baseline + validity flag
    m = monthly.merge(
        pre_stats[['city', 'point_id', 'coh_pre_mean', 'coh_pre_std_safe',
                   'valid_for_monitoring']],
        on=['city', 'point_id'], how='inner')

    # CCD per-month criteria (paper Eq. 3)
    m['delta_gamma'] = m['coh_t'] - m['coh_pre_mean']
    m['z_t'] = m['delta_gamma'] / m['coh_pre_std_safe']
    m['damage_month'] = ((m['delta_gamma'] < k) &
                         (m['z_t'] < z_thresh) &
                         m['valid_for_monitoring'])

    # ----- vectorised consecutive-month run length per (city, point_id) -----
    m = m.sort_values(['city', 'point_id', 'year_month']).reset_index(drop=True)
    # integer month index for arithmetic
    m['ym_int'] = pd.PeriodIndex(m['year_month'], freq='M').asi8
    # within each (city, point_id) group, diff between consecutive month indices
    m['ym_diff'] = m.groupby(['city', 'point_id'])['ym_int'].diff()
    m['ym_diff'] = m['ym_diff'].fillna(99)  # first row in group: gap = sentinel
    # previous row's damage flag within the same group
    prev_dmg = m.groupby(['city', 'point_id'])['damage_month'].shift(1)
    m['prev_damage'] = prev_dmg.fillna(False).astype(bool)
    # a row "starts a new run" if it is a damage month AND it does not continue
    # an immediately-preceding damage month (continues = prev_damage AND ym_diff == 1)
    m['continues_run'] = (m['damage_month'] & m['prev_damage'] &
                          (m['ym_diff'] == 1))
    m['starts_run'] = m['damage_month'] & ~m['continues_run']
    # cumulative count of runs within each (city, point_id)
    m['run_id'] = m.groupby(['city', 'point_id'])['starts_run'].cumsum()
    # run lengths: count damage rows per (city, point_id, run_id)
    runs = m[m['damage_month']].groupby(['city', 'point_id', 'run_id']).size().reset_index(name='run_length')
    if len(runs) > 0:
        max_run = runs.groupby(['city', 'point_id'])['run_length'].max().reset_index()
        max_run.columns = ['city', 'point_id', 'max_run_months']
    else:
        max_run = pd.DataFrame({'city': [], 'point_id': [], 'max_run_months': []})

    # total damage months (regardless of run structure)
    counts = m.groupby(['city', 'point_id'])['damage_month'].sum().reset_index()
    counts.columns = ['city', 'point_id', 'count_damage_months']

    # number of post months observed (denominator-style diagnostic)
    n_obs_months = m.groupby(['city', 'point_id'])['year_month'].nunique().reset_index()
    n_obs_months.columns = ['city', 'point_id', 'n_post_months']

    # assemble output for all pixels with >=1 post observation
    out = max_run.merge(counts, on=['city', 'point_id'], how='outer')
    out = out.merge(n_obs_months, on=['city', 'point_id'], how='outer')

    # include all valid pixels even those with zero post observations (count=0)
    all_valid = pre_stats[pre_stats['valid_for_monitoring']][['city', 'point_id']].drop_duplicates()
    out = all_valid.merge(out, on=['city', 'point_id'], how='left')
    out['max_run_months'] = out['max_run_months'].fillna(0).astype(int)
    out['count_damage_months'] = out['count_damage_months'].fillna(0).astype(int)
    out['n_post_months'] = out['n_post_months'].fillna(0).astype(int)
    out['binary_persistence'] = (out['max_run_months'] >= persistence_months).astype(int)

    return out, m


def evaluate_ccd_results(ccd_out, pol_label, k, z_thresh, persistence_months,
                         source_parquet):
    '''Score CCD output against UNOSAT damage labels. Logs to registry,
    saves OOF parquets for both the continuous count score and the
    binary persistence flag.'''
    out = ccd_out.merge(df_points[['point_id', 'city', TARGET_COL]].drop_duplicates(),
                        on=['point_id', 'city'], how='inner')
    if len(out) < 50 or out[TARGET_COL].nunique() < 2:
        print(f"  CCD {pol_label}: insufficient labelled data after validity mask "
              f"({len(out)} points)")
        return None

    y = out[TARGET_COL].values
    n_dam = int((y == 1).sum())
    n_und = int((y == 0).sum())
    score_count = out['count_damage_months'].values.astype(float)
    score_run = out['max_run_months'].values.astype(float)
    score_binary = out['binary_persistence'].values.astype(int)

    # AUC over continuous scores
    auc_count = (roc_auc_score(y, score_count)
                 if len(np.unique(score_count)) > 1 else float('nan'))
    auc_run = (roc_auc_score(y, score_run)
               if len(np.unique(score_run)) > 1 else float('nan'))

    # paper-protocol binary metrics from persistence flag
    tp = int(((score_binary == 1) & (y == 1)).sum())
    fp = int(((score_binary == 1) & (y == 0)).sum())
    fn = int(((score_binary == 0) & (y == 1)).sum())
    tn = int(((score_binary == 0) & (y == 0)).sum())
    tpr = tp / (tp + fn) if (tp + fn) > 0 else float('nan')
    fpr = fp / (fp + tn) if (fp + tn) > 0 else float('nan')
    precision = tp / (tp + fp) if (tp + fp) > 0 else float('nan')
    f1 = 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) > 0 else float('nan')

    print(f"  CCD-{pol_label} faithful (k={k}, z<{z_thresh}, "
          f"persistence>={persistence_months}m):")
    print(f"    Total points (after validity mask): {len(out):,} (dam={n_dam}, und={n_und})")
    print(f"    AUC(count_damage_months): {auc_count:.3f}")
    print(f"    AUC(max_run_months):      {auc_run:.3f}")
    print(f"    Binary persistence flag (paper protocol):")
    print(f"      TPR={tpr:.3f}  FPR={fpr:.3f}  precision={precision:.3f}  F1={f1:.3f}")
    print(f"      TP={tp} FP={fp} FN={fn} TN={tn}")

    # log continuous score (count_damage_months) as the headline AUC entry
    exp_name = f'B2_CCD_{pol_label}_faithful'
    EXPERIMENT_LOG[exp_name] = {
        'auc': auc_count, 'f1': f1, 'tpr': tpr, 'fpr': fpr,
        'precision': precision, 'n_features': 1, 'method': 'unsupervised_ccd',
        'auc_run': auc_run, 'n_points': len(out),
    }
    ALL_SCORES[f'ccd_{pol_label}'] = pd.DataFrame({
        'point_id': out['point_id'].values,
        'city': out['city'].values,
        f'score_ccd_{pol_label}': score_count,
    })
    registry.log_experiment(
        cell_id='cell_b2', experiment_name=exp_name,
        parquet_name=source_parquet, tier_selection=_tiers,
        classifier_name='unsupervised_ccd',
        feature_set_name=f'ccd_{pol_label}_count_damage_months',
        feature_cols=[f's1__coh_{pol_label}'],
        cv_method='unsupervised', n_folds=0, imputation='none',
        y_true=y, y_proba=score_count, groups=out['city'].values,
        note=(f'Scher 2025 CCD faithful {pol_label} on V3 scene_coh '
              f'(k={k}, z<{z_thresh}, persistence>={persistence_months}m), '
              f'paper TPR={SCHER_PAPER["tpr"]} F1={SCHER_PAPER["f1"]}'),
        tags=['unsupervised', 'ccd', 'scher', 'faithful', pol_label, 'count_score'],
    )
    save_oof_unsupervised(
        point_ids=out['point_id'].values, cities=out['city'].values,
        y_true=y, scores=score_count, model_id=exp_name,
        variant_id=(f'unsupervised_ccd;pol={pol_label};k={k};z={z_thresh};'
                    f'persistence={persistence_months};score=count_damage_months'),
    )

    # also log the binary persistence flag as a separate model_id
    # (paper's actual reported metric is on this binary classifier)
    exp_name_bin = f'B2_CCD_{pol_label}_persistence_faithful'
    EXPERIMENT_LOG[exp_name_bin] = {
        'auc': auc_run, 'f1': f1, 'tpr': tpr, 'fpr': fpr,
        'precision': precision, 'n_features': 1,
        'method': 'unsupervised_ccd_binary',
    }
    save_oof_unsupervised(
        point_ids=out['point_id'].values, cities=out['city'].values,
        y_true=y, scores=score_binary.astype(float), model_id=exp_name_bin,
        variant_id=(f'unsupervised_ccd;pol={pol_label};k={k};z={z_thresh};'
                    f'persistence={persistence_months};score=binary_persistence'),
    )
    return out


# --- load V3 scene_coh ----------------------------------------------
df_sc = load_v3('scene_coh')
if 'period_label' not in df_sc.columns:
    raise RuntimeError(
        "scene_coh V3 missing 'period_label' column. Faithful CCD requires "
        "per-pair period labelling from NB05bV3. Re-run NB05bV3 cell 7 "
        "with FR_SCENE_COH=True to regenerate.")
if 'date' not in df_sc.columns:
    raise RuntimeError(
        "scene_coh V3 missing 'date' column. Expected date column = date2 "
        "(monitoring date of the InSAR pair).")

# verify at least one of the bare polarisation columns is present
have_vv = 's1__coh_vv' in df_sc.columns
have_vh = 's1__coh_vh' in df_sc.columns
if not (have_vv or have_vh):
    raise RuntimeError(
        f"scene_coh V3 missing both 's1__coh_vv' and 's1__coh_vh'. "
        f"Available coh cols: {[c for c in df_sc.columns if 'coh' in c.lower()]}")

# --- diagnostics on raw data ----------------------------------------
print(f"\n  Total scene_coh rows loaded: {len(df_sc):,}")
n_cities = df_sc['city'].nunique()
n_points = df_sc['point_id'].nunique()
print(f"  Unique cities: {n_cities},  unique points: {n_points:,}")
print(f"  period_label distribution:")
for lbl, n in df_sc['period_label'].value_counts().items():
    print(f"    {lbl}: {n:,}")

# --- CCD per polarisation -------------------------------------------
ccd_results = {}
for pol in ('vv', 'vh'):
    val_col = f's1__coh_{pol}'
    if val_col not in df_sc.columns:
        print(f"\n  --- Polarisation {pol.upper()}: column '{val_col}' not in V3 scene_coh, skipping ---")
        continue
    print(f"\n  --- Polarisation {pol.upper()} (column = {val_col}) ---")

    pre_stats = compute_pre_baseline(df_sc, val_col)
    if len(pre_stats) == 0:
        print(f"    No pixels with >=2 prebattle observations; skipping {pol}")
        continue
    print(f"    Pre-conflict baseline: {len(pre_stats):,} points (mean n_pre="
          f"{pre_stats['n_pre'].mean():.1f}, "
          f"mean coh_pre={pre_stats['coh_pre_mean'].mean():.3f}, "
          f"mean coh_std={pre_stats['coh_pre_std'].mean():.3f})")

    monthly = compute_monthly_max_post(df_sc, val_col)
    if len(monthly) == 0:
        print(f"    No post-conflict monthly observations; skipping {pol}")
        continue
    avg_months_per_point = (monthly.groupby(['city', 'point_id'])
                            ['year_month'].nunique().mean())
    print(f"    Post-conflict monthly: {monthly['point_id'].nunique():,} points x avg "
          f"{avg_months_per_point:.1f} months observed")

    ccd_out, monthly_full = apply_ccd_paper(
        pre_stats, monthly,
        k=SCHER_PAPER['k'],
        z_thresh=SCHER_PAPER['z_thresh'],
        persistence_months=SCHER_PAPER['persistence_months'])
    n_valid = int(len(ccd_out))
    n_invalid = len(pre_stats) - n_valid
    valid_pct = 100 * n_valid / max(len(pre_stats), 1)
    print(f"    Validity mask (z_k < {SCHER_PAPER['z_thresh']}): {n_valid:,} valid points "
          f"({valid_pct:.1f}%), {n_invalid:,} excluded as 'unstable'")

    res = evaluate_ccd_results(
        ccd_out, pol,
        k=SCHER_PAPER['k'], z_thresh=SCHER_PAPER['z_thresh'],
        persistence_months=SCHER_PAPER['persistence_months'],
        source_parquet='bda_scene_coh_v3')
    ccd_results[pol] = res

    # free per-pol intermediates
    del pre_stats, monthly, ccd_out, monthly_full
    gc.collect()


# --- combined VV + VH ------------------------------------------------
if ('vv' in ccd_results and 'vh' in ccd_results
        and ccd_results['vv'] is not None and ccd_results['vh'] is not None):
    print(f"\n  --- VV + VH combined ---")
    vv_subset = ccd_results['vv'][['point_id', 'city', TARGET_COL,
                                    'count_damage_months', 'binary_persistence']].rename(
        columns={'count_damage_months': 'count_vv',
                 'binary_persistence': 'bin_vv'})
    vh_subset = ccd_results['vh'][['point_id', 'city',
                                    'count_damage_months', 'binary_persistence']].rename(
        columns={'count_damage_months': 'count_vh',
                 'binary_persistence': 'bin_vh'})
    merged = vv_subset.merge(vh_subset, on=['point_id', 'city'], how='inner')
    if len(merged) > 50 and merged[TARGET_COL].nunique() > 1:
        y = merged[TARGET_COL].values
        sum_count = (merged['count_vv'] + merged['count_vh']).values.astype(float)
        or_persistence = ((merged['bin_vv'] == 1) | (merged['bin_vh'] == 1)).astype(int).values
        auc_sum = (roc_auc_score(y, sum_count)
                   if len(np.unique(sum_count)) > 1 else float('nan'))
        # binary metrics for OR persistence
        tp = int(((or_persistence == 1) & (y == 1)).sum())
        fp = int(((or_persistence == 1) & (y == 0)).sum())
        fn = int(((or_persistence == 0) & (y == 1)).sum())
        tn = int(((or_persistence == 0) & (y == 0)).sum())
        tpr = tp / (tp + fn) if (tp + fn) > 0 else float('nan')
        fpr = fp / (fp + tn) if (fp + tn) > 0 else float('nan')
        precision = tp / (tp + fp) if (tp + fp) > 0 else float('nan')
        f1_or = (2 * tp / (2 * tp + fp + fn)
                 if (2 * tp + fp + fn) > 0 else float('nan'))
        print(f"    VV+VH combined (sum count): n={len(merged):,}  AUC={auc_sum:.3f}")
        print(f"    OR-persistence: TPR={tpr:.3f}  FPR={fpr:.3f}  "
              f"precision={precision:.3f}  F1={f1_or:.3f}")
        EXPERIMENT_LOG['B2_CCD_VV+VH_faithful'] = {
            'auc': auc_sum, 'f1': f1_or, 'tpr': tpr, 'fpr': fpr,
            'precision': precision, 'n_features': 2,
            'method': 'unsupervised_ccd_combined',
        }
        registry.log_experiment(
            cell_id='cell_b2', experiment_name='B2_CCD_VV+VH_faithful',
            parquet_name='bda_scene_coh_v3', tier_selection=_tiers,
            classifier_name='unsupervised_ccd',
            feature_set_name='ccd_vv+vh',
            feature_cols=['s1__coh_vv', 's1__coh_vh'],
            cv_method='unsupervised', n_folds=0, imputation='none',
            y_true=y, y_proba=sum_count, groups=merged['city'].values,
            note=('Scher 2025 CCD VV+VH combined '
                  '(sum of monthly damage counts), faithful'),
            tags=['unsupervised', 'ccd', 'scher', 'faithful', 'combined'],
        )
        save_oof_unsupervised(
            point_ids=merged['point_id'].values, cities=merged['city'].values,
            y_true=y, scores=sum_count, model_id='B2_CCD_VV+VH_faithful',
            variant_id='unsupervised_ccd;pol=vv+vh;score=sum_count_damage_months',
        )

print(f"\n  Scher paper reference (UNOSAT test partition): "
      f"TPR={SCHER_PAPER['tpr']}, FPR={SCHER_PAPER['fpr']}, F1={SCHER_PAPER['f1']}")
print(f"  Note: paper's FPR is computed against a 2019 counterfactual period; "
      f"our FPR above is computed against sampled negatives in the same monitoring "
      f"period and is not directly comparable.")

del df_sc
gc.collect()


CELL B2: CCD FAITHFUL (Scher 2025) -- V3 scene_coh
  load_v3('scene_coh'): 468,995 rows, 11 cols, 18 cities

  Total scene_coh rows loaded: 468,995
  Unique cities: 18,  unique points: 60,769
  period_label distribution:
    prebattle: 202,048
    crossbattle: 186,669
    postbattle: 80,278

  --- Polarisation VV (column = s1__coh_vv) ---
    Pre-conflict baseline: 42,229 points (mean n_pre=3.9, mean coh_pre=0.630, mean coh_std=0.122)
    Post-conflict monthly: 48,404 points x avg 3.4 months observed
    Validity mask (z_k < -2.0): 42,061 valid points (99.6%), 168 excluded as 'unstable'
  CCD-vv faithful (k=-0.05, z<-2.0, persistence>=3m):
    Total points (after validity mask): 42,061 (dam=5485, und=36576)
    AUC(count_damage_months): 0.512
    AUC(max_run_months):      0.512
    Binary persistence flag (paper protocol):
      TPR=0.001  FPR=0.002  precision=0.105  F1=0.003
      TP=8 FP=68 FN=5477 TN=36508
  REG: B2_CCD_vv_faithful                            AUC=0.5119 F1=0.1671 n=4

0

In [8]:
# @title CELL B2_bldg: CCD BUILDING-LEVEL AGGREGATION (max-vote)
print("=" * 70)
print("CELL B2_bldg: CCD BUILDING-LEVEL AGGREGATION (max-vote)")
print("=" * 70)
print("  Aggregating per-pixel Scher CCD predictions to per-building")
print("  footprint via Overture building_id. Paper aggregates per-pixel")
print("  binary persistence flags to settlement level via GHS-BUILT-S;")
print("  we use Overture footprints as the equivalent unit. max() = OR")
print("  for binary flags, sum/max for count_damage_months.")
print()


# --- per polarisation: count_damage_months + binary persistence -----
for pol in ('vv', 'vh'):
    if pol not in ccd_results or ccd_results[pol] is None:
        print(f"  --- {pol.upper()}: no per-pixel results, skipping ---")
        continue
    ccd_df = ccd_results[pol]
    print(f"  --- {pol.upper()} count_damage_months ---")
    evaluate_building_level(
        point_ids=ccd_df['point_id'].values,
        cities=ccd_df['city'].values,
        y_true=ccd_df[TARGET_COL].values,
        scores=ccd_df['count_damage_months'].values.astype(float),
        exp_name_pixel=f'B2_CCD_{pol}_faithful',
        exp_name_bldg=f'B2_CCD_{pol}_faithful_bldg',
        source_parquet='bda_scene_coh_v3',
        classifier_name='unsupervised_ccd',
        cell_id='cell_b2_bldg',
        tags=['unsupervised', 'ccd', 'scher', 'faithful', pol, 'count_score'],
        note_extra=f' (paper F1={SCHER_PAPER["f1"]})',
    )
    print(f"  --- {pol.upper()} binary_persistence (OR over pixels) ---")
    evaluate_building_level(
        point_ids=ccd_df['point_id'].values,
        cities=ccd_df['city'].values,
        y_true=ccd_df[TARGET_COL].values,
        scores=ccd_df['binary_persistence'].values.astype(float),
        exp_name_pixel=f'B2_CCD_{pol}_persistence_faithful',
        exp_name_bldg=f'B2_CCD_{pol}_persistence_faithful_bldg',
        source_parquet='bda_scene_coh_v3',
        classifier_name='unsupervised_ccd',
        cell_id='cell_b2_bldg',
        tags=['unsupervised', 'ccd', 'scher', 'faithful', pol,
              'binary_persistence'],
    )

# --- VV+VH combined --------------------------------------------------
if ('vv' in ccd_results and 'vh' in ccd_results
        and ccd_results['vv'] is not None and ccd_results['vh'] is not None):
    print(f"  --- VV+VH combined sum(count_damage_months) ---")
    vv_part = ccd_results['vv'][['point_id', 'city', TARGET_COL,
                                  'count_damage_months']].rename(
        columns={'count_damage_months': 'count_vv'})
    vh_part = ccd_results['vh'][['point_id', 'city',
                                  'count_damage_months']].rename(
        columns={'count_damage_months': 'count_vh'})
    merged = vv_part.merge(vh_part, on=['point_id', 'city'], how='inner')
    if len(merged) > 50:
        sum_count = (merged['count_vv'] + merged['count_vh']).values.astype(float)
        evaluate_building_level(
            point_ids=merged['point_id'].values,
            cities=merged['city'].values,
            y_true=merged[TARGET_COL].values,
            scores=sum_count,
            exp_name_pixel='B2_CCD_VV+VH_faithful',
            exp_name_bldg='B2_CCD_VV+VH_faithful_bldg',
            source_parquet='bda_scene_coh_v3',
            classifier_name='unsupervised_ccd',
            cell_id='cell_b2_bldg',
            tags=['unsupervised', 'ccd', 'scher', 'faithful', 'combined'],
        )

print(f"\n  Scher paper reference (UNOSAT test): "
      f"TPR={SCHER_PAPER['tpr']}, FPR={SCHER_PAPER['fpr']}, "
      f"F1={SCHER_PAPER['f1']}")


CELL B2_bldg: CCD BUILDING-LEVEL AGGREGATION (max-vote)
  Aggregating per-pixel Scher CCD predictions to per-building
  footprint via Overture building_id. Paper aggregates per-pixel
  binary persistence flags to settlement level via GHS-BUILT-S;
  we use Overture footprints as the equivalent unit. max() = OR
  for binary flags, sum/max for count_damage_months.

  --- VV count_damage_months ---
  B2_CCD_vv_faithful_bldg                      : n_bldg=38,139 (dam=4603/und=33536)  avg_pix/bldg=1.1  AUC(max)=0.521  AUC(mean)=0.519  best=0.521 (max)
  REG: B2_CCD_vv_faithful_bldg                       AUC=0.5205 F1=0.1709 n=38139 feat=1 cities=12 [NB10b_v3/cell_b2_bldg]
    saved oof -> oof_B2_CCD_vv_faithful_bldg__20260507_234413_32b992.parquet  TP=952 FN=3651 FP=5585 TN=27951
  --- VV binary_persistence (OR over pixels) ---
  B2_CCD_vv_persistence_faithful_bldg          : n_bldg=38,139 (dam=4603/und=33536)  avg_pix/bldg=1.1  AUC(max)=0.501  AUC(mean)=0.501  best=0.501 (max)
  REG: B2_CCD_

# Section 2 Optuna: Scher 2025 CCD with Optuna over (k, z_thresh, persistence_months)

The paper fixes `(k, z_thresh, persistence_months) = (-0.05, -2.0, 3)`
from a sensitivity analysis on the 50% UNOSAT calibration half. With 21
cities under GroupKFold there is no clean calibration/test split that
preserves all 21 cities, so this cell reports the **all-21-cities
Optuna AUC** (overfit to 21-city set by definition -- upper bound) and
side-by-side comparison with the paper-faithful B2_bldg numbers above.

Search bounds: `k in [-0.30, 0.0]`, `z_thresh in [-3.0, -1.0]`,
`persistence_months in [1, 6]`. Paper values are inside the range.

Re-uses the `compute_pre_baseline`, `compute_monthly_max_post`,
`apply_ccd_paper` helpers defined in B2 above. Optuna varies only the
threshold and persistence step over the cached arrays.

Both per-pixel and per-building OOF parquets are saved via
`save_oof_unsupervised()`, plus a `registry.log_experiment` entry per
polarisation.

Trial budget: `OPTUNA_TRIALS_FAITHFUL = 200` per polarisation (= 400
evaluations total, cheap because pre/monthly are precomputed).

In [9]:
# @title CELL B2_Optuna: SCHER CCD WITH OPTUNA (k, z_thresh, persistence_months)
print("=" * 70)
print(f"CELL B2_Optuna: SCHER CCD with Optuna ({OPTUNA_TRIALS_FAITHFUL} trials)")
print("=" * 70)
print("  Tuning (k, z_thresh, persistence_months) on V3 scene_coh,")
print("  scoring building-level AUC after max-vote aggregation.")
print(f"  Paper values: k={SCHER_PAPER['k']}, z_thresh={SCHER_PAPER['z_thresh']}, "
      f"persistence={SCHER_PAPER['persistence_months']}")
print()


# Reload V3 scene_coh once and cache outside the objective
_df_sc_b2opt = load_v3('scene_coh')
if 'period_label' not in _df_sc_b2opt.columns:
    raise RuntimeError("scene_coh V3 missing 'period_label' column.")

# Precompute pre baseline + monthly max ONCE per polarisation, since they
# do not depend on (k, z_thresh, persistence_months). Optuna only varies
# the threshold / persistence step.
_pre_stats_cache = {}
_monthly_cache = {}
for _pol in ('vv', 'vh'):
    _vc = f's1__coh_{_pol}'
    if _vc not in _df_sc_b2opt.columns:
        continue
    _pre_stats_cache[_pol] = compute_pre_baseline(_df_sc_b2opt, _vc)
    _monthly_cache[_pol] = compute_monthly_max_post(_df_sc_b2opt, _vc)
    print(f"  Cached: {_pol.upper()} pre_baseline={len(_pre_stats_cache[_pol]):,} pts, "
          f"monthly={len(_monthly_cache[_pol]):,} obs")
print()


def _ccd_objective(trial, pol):
    """One Optuna trial: rebuild CCD with proposed (k, z, persistence),
    aggregate to building level, return building-level AUC. The pre-baseline
    z_k validity mask is recomputed for each k (it depends on k)."""
    k = trial.suggest_float('k', -0.30, 0.0)
    z_thresh = trial.suggest_float('z_thresh', -3.0, -1.0)
    persistence = trial.suggest_int('persistence_months', 1, 6)
    if pol not in _pre_stats_cache:
        return 0.0
    pre_stats = _pre_stats_cache[pol]
    monthly = _monthly_cache[pol]
    if len(pre_stats) == 0 or len(monthly) == 0:
        return 0.0
    ccd_out, _ = apply_ccd_paper(
        pre_stats, monthly, k=k, z_thresh=z_thresh,
        persistence_months=persistence)
    out = ccd_out.merge(
        df_points[['point_id', 'city', TARGET_COL]].drop_duplicates(),
        on=['point_id', 'city'], how='inner')
    if len(out) < 50 or out[TARGET_COL].nunique() < 2:
        return 0.0
    agg = aggregate_to_building(
        point_ids=out['point_id'].values,
        cities=out['city'].values,
        y_true=out[TARGET_COL].values,
        scores=out['count_damage_months'].values.astype(float),
    )
    if agg is None or agg['y_true_max'].nunique() < 2:
        return 0.0
    return float(roc_auc_score(agg['y_true_max'].values,
                                agg['score_max'].values))


B2_OPTUNA_RESULTS = {}
for pol in ('vv', 'vh'):
    if pol not in _pre_stats_cache:
        print(f"  --- {pol.upper()}: not in cache, skipping ---")
        continue
    print(f"  --- {pol.upper()} Optuna ({OPTUNA_TRIALS_FAITHFUL} trials) ---")
    sampler = TPESampler(seed=RANDOM_STATE)
    pruner = MedianPruner(n_startup_trials=10, n_warmup_steps=0)
    study = optuna.create_study(
        direction='maximize', sampler=sampler, pruner=pruner,
        study_name=f'B2_CCD_{pol}_Optuna')
    t0 = time.time()
    study.optimize(lambda t: _ccd_objective(t, pol),
                    n_trials=OPTUNA_TRIALS_FAITHFUL,
                    show_progress_bar=False)
    elapsed = time.time() - t0
    bp = study.best_params
    print(f"    best AUC (bldg-level) = {study.best_value:.4f}  "
          f"({elapsed/60:.1f} min)")
    print(f"    best params: k={bp['k']:.4f}  z_thresh={bp['z_thresh']:.3f}  "
          f"persistence={bp['persistence_months']}")
    print(f"    paper params: k={SCHER_PAPER['k']:.4f}  "
          f"z_thresh={SCHER_PAPER['z_thresh']:.3f}  "
          f"persistence={SCHER_PAPER['persistence_months']}")

    # Re-run with best params to get full OOF for logging
    pre_stats = _pre_stats_cache[pol]
    monthly = _monthly_cache[pol]
    ccd_best, _ = apply_ccd_paper(
        pre_stats, monthly,
        k=bp['k'], z_thresh=bp['z_thresh'],
        persistence_months=bp['persistence_months'])
    out_best = ccd_best.merge(
        df_points[['point_id', 'city', TARGET_COL]].drop_duplicates(),
        on=['point_id', 'city'], how='inner')
    score_best = out_best['count_damage_months'].values.astype(float)
    auc_pixel = (roc_auc_score(out_best[TARGET_COL].values, score_best)
                 if out_best[TARGET_COL].nunique() >= 2 else float('nan'))

    exp_name = f'B2_CCD_{pol}_Optuna'
    EXPERIMENT_LOG[exp_name] = {
        'auc': study.best_value, 'f1': np.nan, 'n_features': 1,
        'method': 'unsupervised_ccd_optuna', 'best_params': bp,
        'aggregation': 'max_vote_building', 'auc_pixel': auc_pixel,
        'n_trials': OPTUNA_TRIALS_FAITHFUL, 'elapsed_min': elapsed / 60,
    }
    registry.log_experiment(
        cell_id='cell_b2_optuna', experiment_name=exp_name,
        parquet_name='bda_scene_coh_v3', tier_selection=_tiers,
        classifier_name='unsupervised_ccd_optuna',
        feature_set_name=f'ccd_{pol}_optuna_bldg',
        feature_cols=[f's1__coh_{pol}'],
        cv_method='unsupervised', n_folds=0, imputation='none',
        y_true=out_best[TARGET_COL].values, y_proba=score_best,
        groups=out_best['city'].values,
        note=(f'Scher 2025 CCD with Optuna over (k, z, persistence) on V3 '
              f'scene_coh, building-level AUC={study.best_value:.4f}, '
              f'best=k{bp["k"]:.3f}_z{bp["z_thresh"]:.2f}_'
              f'p{bp["persistence_months"]}'),
        tags=['unsupervised', 'ccd', 'scher', 'optuna', pol,
              'building_level'],
    )

    # Save per-pixel OOF (in addition to building-level OOF saved below)
    save_oof_unsupervised(
        point_ids=out_best['point_id'].values,
        cities=out_best['city'].values,
        y_true=out_best[TARGET_COL].values,
        scores=score_best,
        model_id=exp_name + '_pixel',
        variant_id=(f'unsupervised_ccd_optuna;pol={pol};'
                    f'k={bp["k"]:.4f};z={bp["z_thresh"]:.3f};'
                    f'persistence={bp["persistence_months"]};'
                    f'aggregation=pixel'),
    )
    # Save building-level OOF
    agg_best = aggregate_to_building(
        point_ids=out_best['point_id'].values,
        cities=out_best['city'].values,
        y_true=out_best[TARGET_COL].values,
        scores=score_best,
    )
    if agg_best is not None and agg_best['y_true_max'].nunique() >= 2:
        save_oof_unsupervised(
            point_ids=agg_best['building_id'].values,
            cities=agg_best['city'].values,
            y_true=agg_best['y_true_max'].values,
            scores=agg_best['score_max'].values,
            model_id=exp_name,
            variant_id=(f'unsupervised_ccd_optuna;pol={pol};'
                        f'k={bp["k"]:.4f};z={bp["z_thresh"]:.3f};'
                        f'persistence={bp["persistence_months"]};'
                        f'aggregation=building_max_vote'),
        )
    B2_OPTUNA_RESULTS[pol] = {
        'study': study, 'best_params': bp,
        'best_auc_bldg': study.best_value,
        'auc_pixel': auc_pixel,
    }

# Compare faithful vs Optuna at building level for VV (headline)
if 'vv' in B2_OPTUNA_RESULTS:
    auc_optuna_vv = B2_OPTUNA_RESULTS['vv']['best_auc_bldg']
    auc_faithful_vv_bldg = EXPERIMENT_LOG.get(
        'B2_CCD_vv_faithful_bldg', {}).get('auc', float('nan'))
    delta = auc_optuna_vv - auc_faithful_vv_bldg
    print(f"\n  VV building-level AUC: faithful={auc_faithful_vv_bldg:.3f}  "
          f"Optuna={auc_optuna_vv:.3f}  delta={delta:+.3f}")

del _df_sc_b2opt, _pre_stats_cache, _monthly_cache
gc.collect()


CELL B2_Optuna: SCHER CCD with Optuna (200 trials)
  Tuning (k, z_thresh, persistence_months) on V3 scene_coh,
  scoring building-level AUC after max-vote aggregation.
  Paper values: k=-0.05, z_thresh=-2.0, persistence=3

  load_v3('scene_coh'): 468,995 rows, 11 cols, 18 cities
  Cached: VV pre_baseline=42,229 pts, monthly=165,979 obs
  Cached: VH pre_baseline=42,229 pts, monthly=165,979 obs

  --- VV Optuna (200 trials) ---
    best AUC (bldg-level) = 0.5240  (5.7 min)
    best params: k=-0.1495  z_thresh=-1.000  persistence=1
    paper params: k=-0.0500  z_thresh=-2.000  persistence=3
  REG: B2_CCD_vv_Optuna                              AUC=0.5114 F1=0.1864 n=42229 feat=1 cities=12 [NB10b_v3/cell_b2_optuna]
    saved oof -> oof_B2_CCD_vv_Optuna_pixel__20260507_234413_32b992.parquet  TP=1528 FN=3971 FP=9364 TN=27366
    saved oof -> oof_B2_CCD_vv_Optuna__20260507_234413_32b992.parquet  TP=1366 FN=3250 FP=8385 TN=25293
  --- VH Optuna (200 trials) ---
    best AUC (bldg-level) = 0.518

0

# Paper 3: Aimaiti 2022 -- Log-ratio of intensity (SAR) + GLCM Mean (Optical)

> Aimaiti, Y., Sanon, C., Koch, M., Baise, L.G., Moaveni, B. (2022). War
> Related Building Damage Assessment in Kyiv, Ukraine, Using Sentinel-1
> Radar and Sentinel-2 Optical Images. Remote Sensing 14(24), 6239.
> Reported on UNOSAT Bucha point inventory: 58% TPR across all damage
> classes; 76% TPR if buildings with footprint >= 300 m^2.

## Method (paper)

Two parallel channels.

**SAR channel (Sentinel-1).** Per pixel, the log ratio of intensity:

  I_ratio = 10 * log10(I_pre / I_post)         (Eq. 1)

with one pre-event scene and one post-event scene. Sentinel-1 GRD
preprocessed in SNAP 8.0 (slice assembly, radiometric calibration to
sigma0, 7x7 Lee Sigma speckle filter, Range Doppler terrain correction).
Per-orbit and per-polarisation thresholds calibrated against UNOSAT on a
54-building AOI (32 intact + 22 damaged):

  Asc VH:  -0.6 / +0.9          Asc VV:  -0.7 / +0.9
  Desc VH: -0.8 / +1.0          Desc VV: -0.65 / +0.9

A pixel is flagged damaged if I_ratio falls below the lower threshold
OR above the upper threshold. Two-tail rule because backscatter can
either decrease (collapse to flat rubble, positive ratio in Eq. 1
sense) or increase (corner-reflector geometry of partial damage,
negative ratio, paper Figure 5(c-2)).

**Optical channel (Sentinel-2).** GLCM Mean texture, 3x3 window,
computed on Sentinel-2 reflectance bands. Pre-event texture =
average of two pre-event scenes (28 March 2021 + 2 January 2022).
A pixel is damaged if texture_post - texture_pre < 0.4.

## In scope for NB10b: B3a (SAR log-ratio)

- V3 `bda_scene_card_t{tier}.parquet` -- one row per (point_id, city,
  date) carrying single-pixel `s1__vv` and `s1__vh`.
- **Single pre / single post selection** matching paper protocol:
  pre  = the latest prebattle date per (city, point_id),
  post = the earliest crossbattle/postbattle date per (city, point_id).
  Paper used 19 Feb 2022 (5 days pre) and 8 April 2022 (~6 weeks post);
  our per-city `battle_start` / `battle_stop` define analogous windows.
- **Log ratio in dB.** SNAP sigma0 calibrated output is in dB after the
  linear-to-dB step in NB03b, so I_ratio_db = I_pre - I_post is
  numerically equivalent to 10*log10 of the linear ratio. A
  `DATA_IS_DB` toggle in the cell switches to 10*log10 if upstream
  ever changes to linear.
- **Two-tail score.** AUC over |I_ratio_db| (matches paper's both-tail
  threshold rule). Signed +I_ratio and -I_ratio also evaluated to pick
  the best AUC direction, mirroring B1's protocol.
- **VV, VH, VV+VH combined** -- VV and VH separately, plus a sum
  |I_ratio_vv| + |I_ratio_vh| OR-style combined score.
- **7x7 Lee Sigma speckle filter** is applied at NB03b preprocessing
  stage, matching the paper's preprocessing step.

## Out of scope for NB10b (documented divergences)

- **Per-orbit (asc/desc) thresholds.** Paper calibrates separate
  thresholds for ascending and descending orbits (4 thresholds total).
  Our V3 `scene_card` is single-orbit per city by data-stack
  convention ("all CARD single-orbit per city" -- one orbit selected
  per city in NB02a, post the v2 cleanup that abolished bonus-city
  multi-orbit clipping). Asc/desc separation cannot be reproduced
  without re-deriving CARD per orbit and tagging each row.
- **Calibrated decision threshold.** Paper sets thresholds from a
  54-building AOI in Bucha. Our reporting uses AUC (threshold-free).
  F1 from a Youden's J cutoff on the same data is reported but is not
  paper-equivalent.
- **300 m^2 footprint filter.** Paper finds TPR rises 58% -> 76% if
  buildings with footprint < 300 m^2 are dropped. Our V3 sample is
  point-based (single-pixel sampling at the building or UNOSAT point);
  footprint area is not currently joined to V3 points. A filter is
  feasible if Overture footprint area is added to bda_points in NB05a
  -- deferred per "do not change NB5b" constraint.
- **OSM / WSF mask.** Paper masks to OSM building footprints or WSF
  built-up to suppress non-building changes (e.g. the Irpin flood).
  Our points are sampled either at UNOSAT positives or at Overture
  building centroids by construction, so the mask is implicit in the
  sample design.
- **OSM-augmented building footprints.** Paper hand-digitises 100
  buildings in the UNOSAT AOI to extend OSM. We use Overture footprints
  globally; no hand-digitisation.

## B3b (GLCM Mean optical) -- OUT OF SCOPE

GLCM Mean texture is computed in raster space with a 3x3 sliding
window on Sentinel-2 reflectance, then averaged across two pre-event
scenes before differencing. NB03e produces per-period spectral indices
(NDVI, BSI, NDBI, etc.) and rolling/zscore products but does NOT
compute GLCM textures. Adding GLCM to NB05b is non-trivial (raster
co-occurrence sweep across multiple bands per scene, then per-point
sampling) and deliberately out-of-scope here per the user's "do not
change NB3e or NB5b" constraint. Aimaiti's paper describes the optical
channel as secondary to the SAR channel ("the texture analysis was
less accurate ... but accurately identified large, damaged buildings");
the SAR replication in B3a is sufficient for paper attribution.

A pseudo-optical baseline computed from V3 spectral-index pre/post
differences (e.g. NDVI_post - NDVI_pre, NDBI_post - NDBI_pre) was
considered and deliberately excluded -- it is a different method and
would not be reportable as Aimaiti's GLCM channel.

## What changed vs NB09b B3a

NB09b B3a operated on V2 zonal-stat means (per-building footprint
aggregation: `s1__vv__mean`). Aimaiti's published method is per-pixel.
NB10b uses V3 `scene_card` single-pixel sampling, matching the paper's
spatial sampling unit. The pre/post selection in NB09b was implicit in
the V2 prepost_card column naming (`pre_vv_mean` aggregated across
all pre dates) -- closer to a mean-of-period than the paper's single-
scene protocol. NB10b explicitly selects single dates per pixel
(latest-pre, earliest-post).

In [10]:
# @title CELL B3a: LOG-RATIO FAITHFUL (Aimaiti 2022) -- V3 per-point SAR
print("=" * 70)
print("CELL B3a: LOG-RATIO FAITHFUL (Aimaiti 2022) -- V3 scene_card")
print("=" * 70)

# Toggle: True if V3 s1__{vv,vh} stores backscatter in dB (SNAP sigma0
# default after the linear-to-dB step in NB03b). False if values are
# in linear power. Diagnostic below prints value range so you can verify.
DATA_IS_DB = True


def select_single_pre_single_post(scene_df, val_col):
    """Per pixel: pick the latest prebattle date and the earliest
    crossbattle/postbattle date. Matches Aimaiti's protocol of using a
    single pre-event and a single post-event scene per location.

    Returns DataFrame with columns city, point_id, I_pre, I_post,
    date_pre, date_post.
    """
    df = scene_df[['city', 'point_id', 'period_label', 'date', val_col]].copy()
    df = df.dropna(subset=[val_col])
    if len(df) == 0:
        return None
    pre = df[df['period_label'] == 'prebattle']
    post = df[df['period_label'].isin(['crossbattle', 'postbattle'])]
    if len(pre) == 0 or len(post) == 0:
        return None
    pre_idx = pre.groupby(['city', 'point_id'])['date'].idxmax()
    pre_pick = pre.loc[pre_idx, ['city', 'point_id', val_col, 'date']]
    pre_pick = pre_pick.rename(columns={val_col: 'I_pre', 'date': 'date_pre'})
    post_idx = post.groupby(['city', 'point_id'])['date'].idxmin()
    post_pick = post.loc[post_idx, ['city', 'point_id', val_col, 'date']]
    post_pick = post_pick.rename(columns={val_col: 'I_post', 'date': 'date_post'})
    merged = pre_pick.merge(post_pick, on=['city', 'point_id'], how='inner')
    return merged


def compute_log_ratio_db(merged):
    """Aimaiti Eq. 1: I_ratio = 10 * log10(I_pre / I_post). Stored in dB.

    Sign: positive => backscatter decrease pre->post (collapse, flat
    rubble); negative => backscatter increase (partial damage, debris
    forming corner reflectors). Paper uses both tails for thresholding,
    so AUC over |I_ratio| is the primary score.

    If DATA_IS_DB is True, I_pre and I_post are already in dB
    (sigma0 dB), so dB difference equals 10*log10 of the linear ratio.
    If False, compute 10*log10 of linear values directly with a small
    floor to avoid log(0).
    """
    if DATA_IS_DB:
        merged['I_ratio_db'] = merged['I_pre'] - merged['I_post']
    else:
        I_pre_safe = merged['I_pre'].clip(lower=1e-12)
        I_post_safe = merged['I_post'].clip(lower=1e-12)
        merged['I_ratio_db'] = 10 * np.log10(I_pre_safe / I_post_safe)
    return merged


def eval_logratio_v3(scores_df, ratio_col, exp_name, pol_label, source_parquet):
    """Score absolute and signed log ratio against UNOSAT, log to
    registry, save OOF."""
    if scores_df is None or len(scores_df) < 50:
        print(f"  {exp_name}: insufficient data ({0 if scores_df is None else len(scores_df)})")
        return None
    y = scores_df[TARGET_COL].values
    if len(np.unique(y)) < 2:
        print(f"  {exp_name}: only one class")
        return None
    raw = scores_df[ratio_col].values
    scores_abs = np.abs(raw)
    scores_neg = -raw
    scores_pos = raw
    auc_abs = roc_auc_score(y, scores_abs)
    auc_neg = roc_auc_score(y, scores_neg)
    auc_pos = roc_auc_score(y, scores_pos)
    auc_best = max(auc_abs, auc_neg, auc_pos)
    if auc_best == auc_abs:
        direction = "|I_ratio|"
        best_scores = scores_abs
    elif auc_best == auc_neg:
        direction = "-I_ratio"
        best_scores = scores_neg
    else:
        direction = "+I_ratio"
        best_scores = scores_pos
    n_dam = int((y == 1).sum()); n_und = int((y == 0).sum())
    print(f"  {exp_name:<35s} ({pol_label}): n={len(y):>6d} (dam={n_dam}/und={n_und})  "
          f"AUC(|I|)={auc_abs:.3f}  AUC(-I)={auc_neg:.3f}  AUC(+I)={auc_pos:.3f}  "
          f"best={auc_best:.3f} ({direction})")
    EXPERIMENT_LOG[exp_name] = {'auc': auc_best, 'f1': np.nan, 'n_features': 1,
                                 'method': 'unsupervised', 'direction': direction}
    ALL_SCORES[f'logratio_{pol_label}'] = pd.DataFrame({
        'point_id': scores_df['point_id'].values,
        'city': scores_df['city'].values,
        f'score_logratio_{pol_label}': best_scores,
    })
    registry.log_experiment(
        cell_id='cell_b3a', experiment_name=exp_name,
        parquet_name=source_parquet, tier_selection=_tiers,
        classifier_name='unsupervised_logratio',
        feature_set_name=f'logratio_{pol_label}', feature_cols=[ratio_col],
        cv_method='unsupervised', n_folds=0, imputation='none',
        y_true=y, y_proba=best_scores, groups=scores_df['city'].values,
        note=f'Aimaiti 2022 SAR log-ratio {pol_label} on V3 scene_card (faithful), '
             f'paper TPR_all={AIMAITI_PAPER["tpr_all"]} TPR_large={AIMAITI_PAPER["tpr_large"]}',
        tags=['unsupervised', 'logratio', 'aimaiti', 'faithful', pol_label],
    )
    save_oof_unsupervised(
        point_ids=scores_df['point_id'].values,
        cities=scores_df['city'].values,
        y_true=y, scores=best_scores, model_id=exp_name,
        variant_id=f'unsupervised_logratio;pol={pol_label};direction={direction};'
                   f'mode=v3_single_pre_single_post;data_is_db={DATA_IS_DB}',
    )
    return auc_best


# --- load V3 scene_card ----------------------------------------------
df_sc = load_v3('scene_card')

if 'period_label' not in df_sc.columns:
    raise RuntimeError(
        "scene_card V3 missing 'period_label' column. Faithful log-ratio requires "
        "per-date period labelling from NB05bV3.")
if 'date' not in df_sc.columns:
    raise RuntimeError(
        "scene_card V3 missing 'date' column. Single-pre/single-post selection "
        "requires acquisition dates.")
for pol in ('vv', 'vh'):
    expected = f's1__{pol}'
    if expected not in df_sc.columns:
        raise RuntimeError(
            f"scene_card V3 missing column '{expected}'. Available SAR cols: "
            f"{[c for c in df_sc.columns if 's1__' in c]}")

# diagnostic: detect dB vs linear from value ranges (informational only)
sample_n = int(df_sc['s1__vv'].notna().sum())
sample = df_sc['s1__vv'].dropna().sample(min(10000, sample_n), random_state=42)
v_min, v_max, v_med = float(sample.min()), float(sample.max()), float(sample.median())
print(f"  s1__vv value range: [{v_min:.3f}, {v_max:.3f}], median={v_med:.3f}")
if v_min < -50 or v_max < 5:
    print(f"    -> appears to be dB (sigma0)")
elif v_min >= 0 and v_max < 5:
    print(f"    -> appears to be linear power; set DATA_IS_DB=False if needed")
print(f"  DATA_IS_DB toggle: {DATA_IS_DB}")

# --- per (point, polarisation) log ratio ------------------------------
ratio_results = {}
for pol in ('vv', 'vh'):
    vcol = f's1__{pol}'
    print(f"\n  --- Polarisation {pol.upper()} (column = {vcol}) ---")
    paired = select_single_pre_single_post(df_sc, vcol)
    if paired is None or len(paired) == 0:
        print(f"    no points with both prebattle and postbattle observations")
        continue
    paired = compute_log_ratio_db(paired)
    print(f"    paired pre/post points: {len(paired):,}  "
          f"(I_ratio_db median={paired['I_ratio_db'].median():.3f}, "
          f"|I_ratio_db| 95th pct={paired['I_ratio_db'].abs().quantile(0.95):.3f})")
    merged = paired.merge(df_points[['point_id', 'city', TARGET_COL]].drop_duplicates(),
                          on=['point_id', 'city'], how='inner')
    if len(merged) < 50:
        print(f"    only {len(merged)} points with damage labels; skipping eval")
        continue
    eval_logratio_v3(merged, 'I_ratio_db', f'B3a_LogRatio_{pol.upper()}_faithful',
                     pol, 'bda_scene_card_v3')
    ratio_results[pol] = merged

# --- VV + VH combined ------------------------------------------------
if 'vv' in ratio_results and 'vh' in ratio_results:
    print(f"\n  --- VV+VH combined ---")
    vv_part = ratio_results['vv'][['point_id', 'city', TARGET_COL, 'I_ratio_db']].rename(
        columns={'I_ratio_db': 'I_ratio_vv'})
    vh_part = ratio_results['vh'][['point_id', 'city', 'I_ratio_db']].rename(
        columns={'I_ratio_db': 'I_ratio_vh'})
    merged = vv_part.merge(vh_part, on=['point_id', 'city'], how='inner')
    if len(merged) > 50 and merged[TARGET_COL].nunique() >= 2:
        y = merged[TARGET_COL].values
        combined = (np.abs(merged['I_ratio_vv'].values)
                    + np.abs(merged['I_ratio_vh'].values))
        auc_comb = roc_auc_score(y, combined)
        print(f"    LogRatio VV+VH combined: AUC={auc_comb:.3f}  ({len(merged):,} points)")
        EXPERIMENT_LOG['B3a_LogRatio_VV+VH_faithful'] = {
            'auc': auc_comb, 'f1': np.nan, 'n_features': 2,
            'method': 'unsupervised', 'direction': '|I_vv|+|I_vh|',
        }
        registry.log_experiment(
            cell_id='cell_b3a', experiment_name='B3a_LogRatio_VV+VH_faithful',
            parquet_name='bda_scene_card_v3', tier_selection=_tiers,
            classifier_name='unsupervised_logratio',
            feature_set_name='logratio_vv+vh',
            feature_cols=['I_ratio_vv', 'I_ratio_vh'],
            cv_method='unsupervised', n_folds=0, imputation='none',
            y_true=y, y_proba=combined, groups=merged['city'].values,
            note='Aimaiti 2022 SAR log-ratio VV+VH combined |I| sum on V3 scene_card (faithful)',
            tags=['unsupervised', 'logratio', 'aimaiti', 'faithful', 'combined'],
        )
        save_oof_unsupervised(
            point_ids=merged['point_id'].values,
            cities=merged['city'].values,
            y_true=y, scores=combined, model_id='B3a_LogRatio_VV+VH_faithful',
            variant_id='unsupervised_logratio;pol=vv+vh;direction=|I_vv|+|I_vh|;'
                       'mode=v3_single_pre_single_post',
        )

print(f"\n  Aimaiti paper reference: TPR_all={AIMAITI_PAPER['tpr_all']}, "
      f"TPR_large(>={AIMAITI_PAPER['min_footprint_m2']}m^2)={AIMAITI_PAPER['tpr_large']}")
print(f"  Note: paper TPR is computed against the UNOSAT Bucha point inventory "
      f"(145 points) using paper-calibrated per-orbit thresholds; our AUC is "
      f"threshold-free over the {len(_tiers)}-tier UNOSAT set.")

del df_sc
gc.collect()


CELL B3a: LOG-RATIO FAITHFUL (Aimaiti 2022) -- V3 scene_card
  load_v3('scene_card'): 834,134 rows, 8 cols, 21 cities
  s1__vv value range: [-100.000, 21.090], median=-5.289
    -> appears to be dB (sigma0)
  DATA_IS_DB toggle: True

  --- Polarisation VV (column = s1__vv) ---
    paired pre/post points: 62,151  (I_ratio_db median=0.161, |I_ratio_db| 95th pct=4.225)
  B3a_LogRatio_VV_faithful            (vv): n= 62151 (dam=8013/und=54138)  AUC(|I|)=0.504  AUC(-I)=0.501  AUC(+I)=0.499  best=0.504 (|I_ratio|)
  REG: B3a_LogRatio_VV_faithful                      AUC=0.5041 F1=0.2212 n=62151 feat=1 cities=20 [NB10b_v3/cell_b3a]
    saved oof -> oof_B3a_LogRatio_VV_faithful__20260507_234413_32b992.parquet  TP=4024 FN=3989 FP=27052 TN=27086

  --- Polarisation VH (column = s1__vh) ---
    paired pre/post points: 62,151  (I_ratio_db median=0.149, |I_ratio_db| 95th pct=4.397)
  B3a_LogRatio_VH_faithful            (vh): n= 62151 (dam=8013/und=54138)  AUC(|I|)=0.499  AUC(-I)=0.500  AUC(+I)=0.500

0

In [11]:
# @title CELL B3a_bldg: LOG-RATIO BUILDING-LEVEL AGGREGATION (max-vote)
print("=" * 70)
print("CELL B3a_bldg: LOG-RATIO BUILDING-LEVEL AGGREGATION (max-vote)")
print("=" * 70)
print("  Aggregating per-pixel Aimaiti log-ratio scores to per-building")
print("  footprint via Overture building_id. Paper protocol: a building")
print("  is damaged if ANY pixel inside its OSM footprint passes")
print("  threshold. max-vote per building_id reproduces this exactly.")
print()


def _logratio_direction_to_score(rdf, exp_name):
    """Recover the AUC-best direction recorded by eval_logratio_v3 and
    return the corresponding score column."""
    info = EXPERIMENT_LOG.get(exp_name, {})
    direction = info.get('direction', '|I_ratio|')
    raw = rdf['I_ratio_db'].values
    if direction == '|I_ratio|':
        return np.abs(raw), direction
    if direction == '-I_ratio':
        return -raw, direction
    return raw, direction


# --- per polarisation -----------------------------------------------
for pol in ('vv', 'vh'):
    if pol not in ratio_results or ratio_results[pol] is None:
        print(f"  --- {pol.upper()}: no per-pixel results, skipping ---")
        continue
    rdf = ratio_results[pol]
    score, direction = _logratio_direction_to_score(
        rdf, f'B3a_LogRatio_{pol.upper()}_faithful')
    print(f"  --- {pol.upper()} (per-pixel direction = {direction}) ---")
    evaluate_building_level(
        point_ids=rdf['point_id'].values,
        cities=rdf['city'].values,
        y_true=rdf[TARGET_COL].values,
        scores=score,
        exp_name_pixel=f'B3a_LogRatio_{pol.upper()}_faithful',
        exp_name_bldg=f'B3a_LogRatio_{pol.upper()}_faithful_bldg',
        source_parquet='bda_scene_card_v3',
        classifier_name='unsupervised_logratio',
        cell_id='cell_b3a_bldg',
        tags=['unsupervised', 'logratio', 'aimaiti', 'faithful', pol],
        note_extra=(f' (paper TPR_all={AIMAITI_PAPER["tpr_all"]}, '
                    f'TPR_large={AIMAITI_PAPER["tpr_large"]})'),
    )

# --- VV+VH combined --------------------------------------------------
if 'vv' in ratio_results and 'vh' in ratio_results:
    print(f"  --- VV+VH combined (|I_vv| + |I_vh|) ---")
    vv_part = ratio_results['vv'][['point_id', 'city', TARGET_COL,
                                    'I_ratio_db']].rename(
        columns={'I_ratio_db': 'I_ratio_vv'})
    vh_part = ratio_results['vh'][['point_id', 'city',
                                    'I_ratio_db']].rename(
        columns={'I_ratio_db': 'I_ratio_vh'})
    merged = vv_part.merge(vh_part, on=['point_id', 'city'], how='inner')
    if len(merged) > 50:
        combined = (np.abs(merged['I_ratio_vv'].values)
                    + np.abs(merged['I_ratio_vh'].values))
        evaluate_building_level(
            point_ids=merged['point_id'].values,
            cities=merged['city'].values,
            y_true=merged[TARGET_COL].values,
            scores=combined,
            exp_name_pixel='B3a_LogRatio_VV+VH_faithful',
            exp_name_bldg='B3a_LogRatio_VV+VH_faithful_bldg',
            source_parquet='bda_scene_card_v3',
            classifier_name='unsupervised_logratio',
            cell_id='cell_b3a_bldg',
            tags=['unsupervised', 'logratio', 'aimaiti', 'faithful',
                  'combined'],
        )

print(f"\n  Aimaiti paper reference: TPR_all={AIMAITI_PAPER['tpr_all']}, "
      f"TPR_large(>={AIMAITI_PAPER['min_footprint_m2']}m^2)="
      f"{AIMAITI_PAPER['tpr_large']}")


CELL B3a_bldg: LOG-RATIO BUILDING-LEVEL AGGREGATION (max-vote)
  Aggregating per-pixel Aimaiti log-ratio scores to per-building
  footprint via Overture building_id. Paper protocol: a building
  is damaged if ANY pixel inside its OSM footprint passes
  threshold. max-vote per building_id reproduces this exactly.

  --- VV (per-pixel direction = |I_ratio|) ---
  B3a_LogRatio_VV_faithful_bldg                : n_bldg=56,562 (dam=6920/und=49642)  avg_pix/bldg=1.1  AUC(max)=0.513  AUC(mean)=0.507  best=0.513 (max)
  REG: B3a_LogRatio_VV_faithful_bldg                 AUC=0.5134 F1=0.2133 n=56562 feat=1 cities=20 [NB10b_v3/cell_b3a_bldg]
    saved oof -> oof_B3a_LogRatio_VV_faithful_bldg__20260507_234413_32b992.parquet  TP=3548 FN=3372 FP=24734 TN=24908
  --- VH (per-pixel direction = +I_ratio) ---
  B3a_LogRatio_VH_faithful_bldg                : n_bldg=56,562 (dam=6920/und=49642)  avg_pix/bldg=1.1  AUC(max)=0.510  AUC(mean)=0.503  best=0.510 (max)
  REG: B3a_LogRatio_VH_faithful_bldg        

# Paper 4: Ahmad 2024 -- UISEM Ensemble

*To be added in next iteration. F8 fusion in V3 supplies the multimodal
feature pool. Faithful replication requires:*

1. *Forward stepwise feature selection + correlation matrix to reduce to
   ~27 features (paper's best Model 2).*
2. *RF + GB + XGB voting ensemble WITHOUT AdaBoost (Ahmad's actual UISEM).*
3. *Ablation cell adds AdaBoost back (Ahmad Model 1) for direct comparison.*

*Out-of-scope: Ahmad uses sklearn GradientBoostingClassifier; we substitute
LightGBM (~50-100x faster on 600k rows; same gradient-boosting family).
A small sklearn GB run on tier-0 only is included for direct comparison.*

In [12]:
# @title CELL B5: AHMAD UISEM FAITHFUL (Model 2, no AdaBoost) -- V3 F8
print("=" * 70)
print("CELL B5: Ahmad UISEM Model 2 faithful -- V3 F8 fusion_composite_blockstats")
print("=" * 70)
print("  RF + GB(LightGBM substitute) + XGBoost soft voting, NO AdaBoost.")
print(f"  Paper acc={AHMAD_PAPER['overall_acc']} on landcover (32 cities, 5 classes).")
print(f"  We report AUC for binary BDA on 21 UNOSAT cities, GroupKFold.")
print()

# Load V3 F8
df_v3_f8 = load_v3('fusion_composite_blockstats')
print(f"  V3 F8: {len(df_v3_f8):,} rows, {df_v3_f8.shape[1]} cols")

# Identify feature columns (drop leakage + metadata)
_meta_cols_v3f8 = {'point_id', 'building_id', 'city', 'tier', TARGET_COL,
                    'x_utm', 'y_utm', 'label_id'}
_f8_feats = [c for c in df_v3_f8.columns if c not in _meta_cols_v3f8]
_f8_feats = drop_leakage(_f8_feats)
print(f"  F8 features after leakage drop: {len(_f8_feats)}")

# prepare_Xy_from imputes NaN with median (required for sklearn voting)
prep_b5 = prepare_Xy_from(df_v3_f8, _f8_feats)
if prep_b5[0] is None:
    raise RuntimeError("prepare_Xy_from returned None for V3 F8")
X_b5, y_b5, groups_b5, sample_ids_b5, feat_cols_b5 = prep_b5
print(f"  Prepared: X={X_b5.shape}, dam={int((y_b5==1).sum())}, "
      f"und={int((y_b5==0).sum())}, cities={len(np.unique(groups_b5))}")
print()

# --- UISEM Model 2 (no AdaBoost = Ahmad's actual best) ---
print("  --- UISEM Model 2 (no AdaBoost) on full F8 features ---")
uisem_m2 = build_uisem(include_adaboost=False)
res_b5 = evaluate_groupkfold(uisem_m2, X_b5, y_b5, groups_b5,
                               label='B5_UISEM_Model2_full_F8',
                               point_ids=sample_ids_b5)
if res_b5 is not None:
    log_result(res_b5, cell_id='cell_b5',
               parquet_name='bda_fusion_composite_blockstats_v3',
               feature_set_name='full_F8_no_FSS',
               feature_cols=feat_cols_b5,
               classifier_name='UISEM_voting_Model2',
               cv_method='GroupKFold',
               note=(f'Ahmad UISEM Model 2 (RF+GB+XGB soft voting, no '
                     f'AdaBoost) on V3 F8 full features. Paper '
                     f'acc={AHMAD_PAPER["overall_acc"]} on landcover. '
                     f'Forward stepwise to 27 features (paper) skipped: '
                     f'paper FSS was for landcover semantics.'),
               tags=['supervised', 'voting', 'ahmad', 'uisem',
                     'faithful', 'model2', 'v3', 'fusion_composite_blockstats'])
    save_oof_from_result(res_b5, 'B5_UISEM_Model2_full_F8')

# --- Sub-classifier individual AUCs (Ahmad found XGBoost best individually) ---
print(f"\n  --- Sub-classifier individual AUCs ---")
B5_SUB_RESULTS = {}
for name, est in build_uisem(include_adaboost=False).estimators:
    res_sub = evaluate_groupkfold(est, X_b5, y_b5, groups_b5,
                                    label=f'B5_sub_{name.upper()}_full_F8',
                                    point_ids=sample_ids_b5)
    if res_sub is not None:
        log_result(res_sub, cell_id='cell_b5',
                   parquet_name='bda_fusion_composite_blockstats_v3',
                   feature_set_name='full_F8_sub_classifier',
                   feature_cols=feat_cols_b5,
                   classifier_name=name.upper(),
                   cv_method='GroupKFold',
                   note=f'Sub-classifier from UISEM Model 2 (paper '
                        f'reported XGBoost as top individual)',
                   tags=['supervised', 'sub_classifier', 'ahmad', 'uisem',
                         name, 'v3'])
        save_oof_from_result(res_sub, f'B5_sub_{name.upper()}_full_F8')
        B5_SUB_RESULTS[name.upper()] = res_sub

# Summary: voting vs individual
if res_b5 is not None and B5_SUB_RESULTS:
    print(f"\n  --- B5 voting vs individual AUC summary ---")
    print(f"    {'UISEM Model 2 (voting)':<30s}: AUC = {res_b5['auc']:.4f}")
    for name, r in B5_SUB_RESULTS.items():
        print(f"    {name:<30s}: AUC = {r['auc']:.4f}")
    best_sub_name = max(B5_SUB_RESULTS.items(),
                         key=lambda kv: kv[1]['auc'])[0]
    best_sub_auc = B5_SUB_RESULTS[best_sub_name]['auc']
    delta = res_b5['auc'] - best_sub_auc
    print(f"\n    Voting vs best sub ({best_sub_name}): {delta:+.4f}")
    print(f"    (Ahmad: voting +1pp over best sub for landcover)")

# X_b5/y_b5/groups_b5/sample_ids_b5/feat_cols_b5/df_v3_f8 stay in scope
# for B5_bldg and B6 which run after this cell.


CELL B5: Ahmad UISEM Model 2 faithful -- V3 F8 fusion_composite_blockstats
  RF + GB(LightGBM substitute) + XGBoost soft voting, NO AdaBoost.
  Paper acc=0.92 on landcover (32 cities, 5 classes).
  We report AUC for binary BDA on 21 UNOSAT cities, GroupKFold.

  load_v3('fusion_composite_blockstats'): 62,043 rows, 415 cols, 19 cities
  V3 F8: 62,043 rows, 415 cols
  F8 features after leakage drop: 409
  Prepared: X=(62043, 403), dam=8080, und=53963, cities=19

  --- UISEM Model 2 (no AdaBoost) on full F8 features ---
    B5_UISEM_Model2_full_F8                      : AUC=0.650 (+/-0.032)  F1=0.043
  REG: B5_UISEM_Model2_full_F8                       AUC=0.6495 F1=0.0427 n=62043 feat=403 cities=19 [NB10b_v3/cell_b5]
    saved oof -> oof_B5_UISEM_Model2_full_F8__20260507_234413_32b992.parquet  TP=184 FN=7896 FP=353 TN=53610

  --- Sub-classifier individual AUCs ---
    B5_sub_RF_full_F8                            : AUC=0.664 (+/-0.025)  F1=0.023
  REG: B5_sub_RF_full_F8                  

In [13]:
# @title CELL B5_bldg: UISEM BUILDING-LEVEL AGGREGATION
print("=" * 70)
print("CELL B5_bldg: UISEM building-level aggregation (max-vote per building_id)")
print("=" * 70)

if 'res_b5' not in globals() or res_b5 is None:
    print("  res_b5 not in scope. Run B5 first.")
else:
    print(f"  --- B5 UISEM Model 2 building-level ---")
    evaluate_building_level(
        point_ids=res_b5['point_id'],
        cities=res_b5['groups'],
        y_true=res_b5['y_true'],
        scores=res_b5['y_proba'],
        exp_name_pixel='B5_UISEM_Model2_full_F8',
        exp_name_bldg='B5_UISEM_Model2_full_F8_bldg',
        source_parquet='bda_fusion_composite_blockstats_v3',
        classifier_name='UISEM_voting_Model2',
        cell_id='cell_b5_bldg',
        tags=['supervised', 'voting', 'ahmad', 'uisem', 'faithful',
              'model2', 'v3'],
        note_extra=(f' (paper acc={AHMAD_PAPER["overall_acc"]}, '
                    f'paper F1={AHMAD_PAPER["f1"]})'),
    )
    print()

# Aggregate sub-classifiers too -- useful for the comparison table
if 'B5_SUB_RESULTS' in globals():
    for name, r in B5_SUB_RESULTS.items():
        print(f"  --- B5 sub-classifier {name} building-level ---")
        evaluate_building_level(
            point_ids=r['point_id'],
            cities=r['groups'],
            y_true=r['y_true'],
            scores=r['y_proba'],
            exp_name_pixel=f'B5_sub_{name}_full_F8',
            exp_name_bldg=f'B5_sub_{name}_full_F8_bldg',
            source_parquet='bda_fusion_composite_blockstats_v3',
            classifier_name=name,
            cell_id='cell_b5_bldg',
            tags=['supervised', 'sub_classifier', 'ahmad', 'uisem',
                  name.lower(), 'v3'],
        )


CELL B5_bldg: UISEM building-level aggregation (max-vote per building_id)
  --- B5 UISEM Model 2 building-level ---
  B5_UISEM_Model2_full_F8_bldg                 : n_bldg=56,090 (dam=6862/und=49228)  avg_pix/bldg=1.1  AUC(max)=0.668  AUC(mean)=0.664  best=0.668 (max)
  REG: B5_UISEM_Model2_full_F8_bldg                  AUC=0.6678 F1=0.0505 n=56090 feat=1 cities=19 [NB10b_v3/cell_b5_bldg]
    saved oof -> oof_B5_UISEM_Model2_full_F8_bldg__20260507_234413_32b992.parquet  TP=4837 FN=2025 FP=23209 TN=26019

  --- B5 sub-classifier RF building-level ---
  B5_sub_RF_full_F8_bldg                       : n_bldg=56,090 (dam=6862/und=49228)  avg_pix/bldg=1.1  AUC(max)=0.686  AUC(mean)=0.682  best=0.686 (max)
  REG: B5_sub_RF_full_F8_bldg                        AUC=0.6863 F1=0.0287 n=56090 feat=1 cities=19 [NB10b_v3/cell_b5_bldg]
    saved oof -> oof_B5_sub_RF_full_F8_bldg__20260507_234413_32b992.parquet  TP=5024 FN=1838 FP=23022 TN=26206
  --- B5 sub-classifier GB building-level ---
  B5_sub_GB

# CELL B6: AdaBoost ablation (Ahmad Model 1 vs Model 2)

Ahmad's paper compared six models: Model 1 (RF + AdaBoost + GB + XGB)
vs Model 2 (RF + GB + XGB) at three feature counts (27 / 35 / 47). Across
all feature counts the no-AdaBoost variant beat the with-AdaBoost variant
by ~1pp accuracy. Model 2 at 27 features = UISEM = 92% accuracy.

This cell runs the AdaBoost ablation on V3 F8 (Model 1) and compares
to B5's Model 2. Reports both per-pixel and building-level AUCs.

The 27-feature forward stepwise selection variant from the paper is
skipped (would require ~3000 model fits and is paper-specific to land
cover semantics, not transferable to BDA).


In [14]:
# @title CELL B6: AdaBoost ablation (Ahmad Model 1 = with AdaBoost)
print("=" * 70)
print("CELL B6: AdaBoost ablation -- Ahmad Model 1 vs Model 2")
print("=" * 70)
print(f"  Paper finding: Model 2 (no AB) beat Model 1 (with AB) by ~1pp acc.")
print(f"  This cell adds AdaBoost to test the same on BDA.")
print()

if 'X_b5' not in globals() or X_b5 is None:
    print("  X_b5 not in scope. Run B5 first.")
else:
    print("  --- UISEM Model 1 (RF + AdaBoost + GB + XGB) on full F8 ---")
    uisem_m1 = build_uisem(include_adaboost=True)
    res_b6 = evaluate_groupkfold(uisem_m1, X_b5, y_b5, groups_b5,
                                   label='B6_UISEM_Model1_full_F8',
                                   point_ids=sample_ids_b5)
    if res_b6 is not None:
        log_result(res_b6, cell_id='cell_b6',
                   parquet_name='bda_fusion_composite_blockstats_v3',
                   feature_set_name='full_F8_no_FSS',
                   feature_cols=feat_cols_b5,
                   classifier_name='UISEM_voting_Model1',
                   cv_method='GroupKFold',
                   note=(f'Ahmad UISEM Model 1 (RF+AB+GB+XGB) -- AdaBoost '
                         f'ablation control. Paper: Model 2 - Model 1 = +1pp '
                         f'accuracy on landcover.'),
                   tags=['supervised', 'voting', 'ahmad', 'uisem',
                         'ablation', 'model1', 'adaboost', 'v3',
                         'fusion_composite_blockstats'])
        save_oof_from_result(res_b6, 'B6_UISEM_Model1_full_F8')

        # Building-level
        evaluate_building_level(
            point_ids=res_b6['point_id'],
            cities=res_b6['groups'],
            y_true=res_b6['y_true'],
            scores=res_b6['y_proba'],
            exp_name_pixel='B6_UISEM_Model1_full_F8',
            exp_name_bldg='B6_UISEM_Model1_full_F8_bldg',
            source_parquet='bda_fusion_composite_blockstats_v3',
            classifier_name='UISEM_voting_Model1',
            cell_id='cell_b6_bldg',
            tags=['supervised', 'voting', 'ahmad', 'uisem', 'ablation',
                  'model1', 'adaboost', 'v3'],
        )

    # Side-by-side comparison Model 1 vs Model 2
    if res_b5 is not None and res_b6 is not None:
        delta_pixel = res_b5['auc'] - res_b6['auc']
        m2_bldg = EXPERIMENT_LOG.get(
            'B5_UISEM_Model2_full_F8_bldg', {}).get('auc', float('nan'))
        m1_bldg = EXPERIMENT_LOG.get(
            'B6_UISEM_Model1_full_F8_bldg', {}).get('auc', float('nan'))
        delta_bldg = m2_bldg - m1_bldg
        print(f"\n  --- AdaBoost ablation summary ---")
        print(f"    Model 2 (no AB)    pixel AUC = {res_b5['auc']:.4f}  "
              f"bldg AUC = {m2_bldg:.4f}")
        print(f"    Model 1 (with AB)  pixel AUC = {res_b6['auc']:.4f}  "
              f"bldg AUC = {m1_bldg:.4f}")
        print(f"    Delta (M2 - M1)    pixel = {delta_pixel:+.4f}  "
              f"bldg = {delta_bldg:+.4f}")
        print(f"    Paper delta on landcover acc: +1pp (Model 2 wins)")

# Free V3 F8 memory at the end of the F8 section
if 'df_v3_f8' in globals():
    del df_v3_f8
if 'X_b5' in globals():
    del X_b5, y_b5, groups_b5, sample_ids_b5, feat_cols_b5
gc.collect()


CELL B6: AdaBoost ablation -- Ahmad Model 1 vs Model 2
  Paper finding: Model 2 (no AB) beat Model 1 (with AB) by ~1pp acc.
  This cell adds AdaBoost to test the same on BDA.

  --- UISEM Model 1 (RF + AdaBoost + GB + XGB) on full F8 ---
    B6_UISEM_Model1_full_F8                      : AUC=0.657 (+/-0.031)  F1=0.034
  REG: B6_UISEM_Model1_full_F8                       AUC=0.6574 F1=0.0341 n=62043 feat=403 cities=19 [NB10b_v3/cell_b6]
    saved oof -> oof_B6_UISEM_Model1_full_F8__20260507_234413_32b992.parquet  TP=145 FN=7935 FP=269 TN=53694
  B6_UISEM_Model1_full_F8_bldg                 : n_bldg=56,090 (dam=6862/und=49228)  avg_pix/bldg=1.1  AUC(max)=0.676  AUC(mean)=0.673  best=0.676 (max)
  REG: B6_UISEM_Model1_full_F8_bldg                  AUC=0.6762 F1=0.0408 n=56090 feat=1 cities=19 [NB10b_v3/cell_b6_bldg]
    saved oof -> oof_B6_UISEM_Model1_full_F8_bldg__20260507_234413_32b992.parquet  TP=4941 FN=1921 FP=23105 TN=26123

  --- AdaBoost ablation summary ---
    Model 2 (no AB)  

348

# Section 4 Optuna: Ahmad UISEM with per-classifier Optuna + tuned voting

Mirrors NB10a C3 + C4 structure on V3 F8. Per-classifier Optuna at 200
trials each over RF / ExtraTrees / GBM / XGBoost / LightGBM, then
re-build a soft-voting ensemble from the tuned sub-classifiers.

Reports for each classifier:

- Per-pixel tuned AUC (saved via `save_oof_from_result`,
  registered via `log_result`)
- Per-building tuned AUC (saved via `save_oof_unsupervised` and
  `registry.log_experiment` through `evaluate_building_level`)

Plus the tuned voting ensemble on top of all per-classifier-tuned models.

Self-contained: re-loads V3 F8 (B6 above already deleted the in-scope
copy). The B5 baseline numbers are still in `EXPERIMENT_LOG` and used
for the untuned-vs-tuned comparison summary.

Trial budget: `OPTUNA_TRIALS_FAITHFUL = 200` per classifier x 5
classifiers = 1000 total trials. Each trial is a 5-fold GroupKFold
evaluation on ~62k V3 F8 points. Estimated wall time: 30 to 90 minutes.

In [16]:
# @title CELL B5_Optuna: Per-classifier Optuna + tuned voting on V3 F8
# 2026-05-10: GBM dropped, replaced by LightGBM (same algorithm family,
# precedent set in NB09b). See thesis methodology section on classifier-choice.
print("=" * 70)
print(f"CELL B5_Optuna: Per-classifier Optuna ({OPTUNA_TRIALS_FAITHFUL} trials each)")
print("=" * 70)

# --- Self-contained data load (NB11b does not depend on NB10b kernel state) ---
print("  Loading V3 F8 fusion_composite_blockstats...")
df_v3_f8 = load_v3('fusion_composite_blockstats')
_meta_cols_v3f8 = {'point_id', 'building_id', 'city', 'tier', TARGET_COL,
                    'x_utm', 'y_utm', 'label_id'}
_f8_feats = [c for c in df_v3_f8.columns if c not in _meta_cols_v3f8]
_f8_feats = drop_leakage(_f8_feats)
prep_b5opt = prepare_Xy_from(df_v3_f8, _f8_feats)
if prep_b5opt[0] is None:
    raise RuntimeError("prepare_Xy_from returned None for V3 F8")
X_b5, y_b5, groups_b5, sample_ids_b5, feat_cols_b5 = prep_b5opt
print(f"  V3 F8 prepared: X={X_b5.shape}, dam={int((y_b5==1).sum())}, "
      f"und={int((y_b5==0).sum())}, cities={len(np.unique(groups_b5))}")
print()

B5_OPTUNA_RESULTS = {}
# GBM excluded: scikit-learn GradientBoostingClassifier is single-threaded and does
# not scale to V3 F8 (62k samples x 403 feats x 200 trials x 5 folds = 3+ days).
# LightGBM covers the gradient-boosted-trees family. NB09b uses the same swap.
_b5_candidates = [c for c in ['RF', 'ExtraTrees', 'LightGBM',
                                'XGBoost'] if c in SEARCH_SPACES]
print(f"  Candidates: {_b5_candidates}  (GBM excluded -> LightGBM substitute)")
print()

# --- Optional cache reload: if a {clf_name}_BEST OOF is already on disk from a
#     prior partial run, skip re-running Optuna for it. ---
def _try_reload_cached_b5(clf_name):
    """Try to reconstruct an `opt`-shaped dict from a cached OOF parquet for this
    classifier. Returns None if no usable cache is found."""
    try:
        from pathlib import Path
        _oof_root = Path(globals().get('OOF_DIR', globals().get('OUT_DIR', '.')))
        if not _oof_root.exists():
            return None
        _hits = sorted(_oof_root.glob(f'oof_B5_UISEM_{clf_name}_Optuna__*.parquet'))
        if not _hits:
            return None
        import pandas as pd
        from sklearn.metrics import roc_auc_score, f1_score
        df_oof = pd.read_parquet(_hits[-1])
        if not {'y_true', 'y_proba'}.issubset(df_oof.columns):
            return None
        yt = df_oof['y_true'].values
        yp = df_oof['y_proba'].values
        try:
            _auc = roc_auc_score(yt, yp)
        except Exception:
            _auc = float('nan')
        try:
            _f1 = f1_score(yt, (yp >= 0.5).astype(int), zero_division=0)
        except Exception:
            _f1 = float('nan')
        _grp = df_oof['groups'].values if 'groups' in df_oof.columns else df_oof.get('city', pd.Series([''] * len(df_oof))).values
        _pid = df_oof['point_id'].values if 'point_id' in df_oof.columns else df_oof.index.values
        cached_result = {
            'y_true':  yt,  'y_proba': yp,
            'groups':  _grp, 'point_id': _pid,
            'auc':     _auc, 'f1':     _f1,
        }
        return {'best_result': cached_result,
                'best_params': {},
                'cached_from': str(_hits[-1].name)}
    except Exception as _e:
        print(f"    (cache reload failed for {clf_name}: {_e})")
        return None


for clf_name in _b5_candidates:
    _cached = _try_reload_cached_b5(clf_name)
    if _cached is not None:
        print(f"  --- {clf_name}: cached OOF found ({_cached['cached_from']}) "
              f"-> skipping Optuna ---")
        print(f"    cached AUC = {_cached['best_result']['auc']:.4f}  "
              f"F1 = {_cached['best_result']['f1']:.4f}")
        B5_OPTUNA_RESULTS[clf_name] = _cached
        print()
        continue

    print(f"  --- {clf_name} Optuna on V3 F8 ({OPTUNA_TRIALS_FAITHFUL} trials) ---")
    opt = run_optuna_groupkfold(
        classifier_name=clf_name, X=X_b5, y=y_b5, groups=groups_b5,
        n_trials=OPTUNA_TRIALS_FAITHFUL,
        study_name=f'B5_UISEM_{clf_name}_Optuna',
        point_ids=sample_ids_b5)
    if opt['best_result'] is not None:
        log_result(opt['best_result'], cell_id='cell_b5_optuna',
                   parquet_name='bda_fusion_composite_blockstats_v3',
                   feature_set_name='full_F8_optuna',
                   feature_cols=feat_cols_b5,
                   classifier_name=f'{clf_name}-Optuna',
                   cv_method='GroupKFold',
                   note=(f'B5 Optuna {OPTUNA_TRIALS_FAITHFUL} trials, '
                         f'GroupKFold. Best params: {opt["best_params"]}'),
                   tags=['supervised', 'optuna', 'ahmad', 'uisem',
                         clf_name.lower(), 'v3',
                         'fusion_composite_blockstats'])
        save_oof_from_result(opt['best_result'],
                              f'B5_UISEM_{clf_name}_Optuna')
        evaluate_building_level(
            point_ids=opt['best_result']['point_id'],
            cities=opt['best_result']['groups'],
            y_true=opt['best_result']['y_true'],
            scores=opt['best_result']['y_proba'],
            exp_name_pixel=f'B5_UISEM_{clf_name}_Optuna',
            exp_name_bldg=f'B5_UISEM_{clf_name}_Optuna_bldg',
            source_parquet='bda_fusion_composite_blockstats_v3',
            classifier_name=f'{clf_name}-Optuna',
            cell_id='cell_b5_optuna_bldg',
            tags=['supervised', 'optuna', 'ahmad', 'uisem',
                  clf_name.lower(), 'v3'],
        )
        B5_OPTUNA_RESULTS[clf_name] = opt
    print()

# Pick the per-classifier winner
if B5_OPTUNA_RESULTS:
    _best_pair = max(B5_OPTUNA_RESULTS.items(),
                      key=lambda kv: kv[1]['best_result']['auc'])
    _best_clf_name = _best_pair[0]
    _best_clf_auc = _best_pair[1]['best_result']['auc']
    print(f"\n  B5 Optuna per-classifier winner: {_best_clf_name} "
          f"AUC = {_best_clf_auc:.4f}")

# --- Tuned voting ensemble: rebuild VotingClassifier with best params ---
# Skip voting if any constituent was loaded from cache without best_params (we cannot
# rebuild the estimator with empty params).
_voting_eligible = {k: v for k, v in B5_OPTUNA_RESULTS.items()
                    if v.get('best_params')}
if len(_voting_eligible) >= 2:
    print(f"\n  --- Tuned voting ensemble (soft) on V3 F8 "
          f"[{len(_voting_eligible)} of {len(B5_OPTUNA_RESULTS)} constituents have params] ---")
    from sklearn.ensemble import VotingClassifier
    _tuned_estimators = []
    for clf_name, opt in _voting_eligible.items():
        _clf_class = SEARCH_SPACES[clf_name][1]
        _tuned_estimators.append(
            (clf_name.lower(), _clf_class(**opt['best_params'])))
    _tuned_voting = VotingClassifier(estimators=_tuned_estimators,
                                       voting='soft')
    res_b5_tuned = evaluate_groupkfold(
        _tuned_voting, X_b5, y_b5, groups_b5,
        label='B5_UISEM_Tuned_Voting', point_ids=sample_ids_b5)
    if res_b5_tuned is not None:
        log_result(res_b5_tuned, cell_id='cell_b5_optuna',
                   parquet_name='bda_fusion_composite_blockstats_v3',
                   feature_set_name='full_F8_optuna_voting',
                   feature_cols=feat_cols_b5,
                   classifier_name='UISEM_voting_Optuna_tuned',
                   cv_method='GroupKFold',
                   note=(f'Tuned voting ensemble of '
                         f'{len(_tuned_estimators)} per-classifier-Optuna '
                         f'best models. {OPTUNA_TRIALS_FAITHFUL} trials each.'),
                   tags=['supervised', 'voting', 'optuna', 'ahmad',
                         'uisem', 'tuned', 'v3',
                         'fusion_composite_blockstats'])
        save_oof_from_result(res_b5_tuned, 'B5_UISEM_Tuned_Voting')
        evaluate_building_level(
            point_ids=res_b5_tuned['point_id'],
            cities=res_b5_tuned['groups'],
            y_true=res_b5_tuned['y_true'],
            scores=res_b5_tuned['y_proba'],
            exp_name_pixel='B5_UISEM_Tuned_Voting',
            exp_name_bldg='B5_UISEM_Tuned_Voting_bldg',
            source_parquet='bda_fusion_composite_blockstats_v3',
            classifier_name='UISEM_voting_Optuna_tuned',
            cell_id='cell_b5_optuna_voting_bldg',
            tags=['supervised', 'voting', 'optuna', 'ahmad', 'uisem',
                  'tuned', 'v3'],
        )
elif len(B5_OPTUNA_RESULTS) >= 2:
    print(f"\n  (Voting ensemble skipped: {len(B5_OPTUNA_RESULTS)} classifiers but only "
          f"{len(_voting_eligible)} have best_params -- re-run those classifiers without "
          f"cache to rebuild the voting ensemble.)")

# --- Summary: untuned Model 2 vs tuned per-classifier vs tuned voting ---
print(f"\n  --- B5 Optuna summary on V3 F8 ---")
auc_untuned = (res_b5['auc'] if 'res_b5' in globals()
                and res_b5 is not None else float('nan'))
print(f"    Untuned UISEM Model 2 (B5):     AUC = {auc_untuned:.4f}")
for clf_name, opt in B5_OPTUNA_RESULTS.items():
    _src = '(cached)' if not opt.get('best_params') else ''
    print(f"    Tuned {clf_name:<10s} ({OPTUNA_TRIALS_FAITHFUL} trials): AUC = "
          f"{opt['best_result']['auc']:.4f} {_src}")
if 'res_b5_tuned' in globals() and res_b5_tuned is not None:
    print(f"    Tuned voting ensemble:           AUC = "
          f"{res_b5_tuned['auc']:.4f}")

# Free V3 F8 memory
if 'df_v3_f8' in globals():
    del df_v3_f8
if 'X_b5' in globals():
    del X_b5, y_b5, groups_b5, sample_ids_b5, feat_cols_b5
gc.collect()

CELL B5_Optuna: Per-classifier Optuna (200 trials each)
  Loading V3 F8 fusion_composite_blockstats...
  load_v3('fusion_composite_blockstats'): 62,043 rows, 415 cols, 19 cities
  V3 F8 prepared: X=(62043, 403), dam=8080, und=53963, cities=19

  Candidates: ['RF', 'ExtraTrees', 'LightGBM', 'XGBoost']  (GBM excluded -> LightGBM substitute)

  --- RF: cached OOF found (oof_B5_UISEM_RF_Optuna__20260507_234413_32b992.parquet) -> skipping Optuna ---
    cached AUC = 0.6811  F1 = 0.3123

  --- ExtraTrees: cached OOF found (oof_B5_UISEM_ExtraTrees_Optuna__20260507_234413_32b992.parquet) -> skipping Optuna ---
    cached AUC = 0.6826  F1 = 0.3097

  --- LightGBM Optuna on V3 F8 (200 trials) ---
  Optuna LightGBM (200 trials, study=B5_UISEM_LightGBM_Optuna)
    B5_UISEM_LightGBM_Optuna_trial0              : AUC=0.647 (+/-0.031)  F1=0.017
    B5_UISEM_LightGBM_Optuna_trial1              : AUC=0.625 (+/-0.029)  F1=0.123
    B5_UISEM_LightGBM_Optuna_trial2              : AUC=0.666 (+/-0.030)  F1=0

2130

# CELL B8: COMPARISON TABLE

Aggregates all faithful and Optuna results from EXPERIMENT_LOG and the
ResultRegistry. Compares against paper headline numbers.

In [17]:
# @title CELL B8: COMPARISON TABLE
print("=" * 70)
print("CELL B8: NB10b COMPARISON TABLE")
print("=" * 70)

rows = []
for exp_name, info in EXPERIMENT_LOG.items():
    if isinstance(info, dict) and 'auc' in info:
        rows.append({
            'experiment': exp_name,
            'auc': info.get('auc'),
            'f1': info.get('f1'),
            'n_features': info.get('n_features'),
            'method': info.get('method', ''),
            'direction': info.get('direction', ''),
        })

if not rows:
    print("  No experiments logged yet. Run B1+ cells above first.")
else:
    df_cmp = pd.DataFrame(rows).sort_values('auc', ascending=False).reset_index(drop=True)
    print(f"\n  {len(df_cmp)} experiments logged.\n")
    with pd.option_context('display.max_rows', None,
                            'display.max_columns', None,
                            'display.width', 200):
        print(df_cmp.to_string(index=False))

    save_result(df_cmp, 'comparison_table', 'cell_b8', fmt='csv')

    # paper reference comparison
    print(f"\n  Paper reference numbers (faithful replication targets):")
    print(f"    Ballinger PWTT:           AUC={BALLINGER_PAPER['auc']}  F1={BALLINGER_PAPER['f1']}")
    print(f"    Scher CCD (binary):       F1={SCHER_PAPER['f1']}  TPR={SCHER_PAPER['tpr']}")
    print(f"    Aimaiti SAR log-ratio:    TPR={AIMAITI_PAPER['tpr_all']} (all), {AIMAITI_PAPER['tpr_large']} (footprint > {AIMAITI_PAPER['min_footprint_m2']} m^2)")
    print(f"    Ahmad UISEM:              acc={AHMAD_PAPER['overall_acc']}  F1={AHMAD_PAPER['f1']}  precision={AHMAD_PAPER['precision']}")
    print(f"    Dietrich (NB10a, ref):    AUC={DIETRICH_PAPER['auc']}  F1={DIETRICH_PAPER['f1']}")


CELL B8: NB10b COMPARISON TABLE

  44 experiments logged.

                         experiment      auc       f1  n_features                    method     direction
            B5_UISEM_RF_Optuna_bldg 0.705219      NaN           1   unsupervised_aggregated           max
    B5_UISEM_ExtraTrees_Optuna_bldg 0.704643      NaN           1   unsupervised_aggregated           max
      B5_UISEM_LightGBM_Optuna_bldg 0.694946      NaN           1   unsupervised_aggregated           max
         B5_UISEM_Tuned_Voting_bldg 0.693887      NaN           1   unsupervised_aggregated           max
             B5_sub_RF_full_F8_bldg 0.686335      NaN           1   unsupervised_aggregated           max
       B5_UISEM_XGBoost_Optuna_bldg 0.685419      NaN           1   unsupervised_aggregated           max
    B5_UISEM_ExtraTrees_Optuna_BEST 0.682611 0.309744         403                                        
            B5_UISEM_RF_Optuna_BEST 0.681138 0.312302         403                            